# Jour 4 · Speech-to-speech, évaluation et production

**Formation Agents vocaux de 0 à Z · AI&ML Factory**

Les trois premiers jours ont construit Awa **en cascade** : l'oreille (STT), le cerveau (LLM) et la voix (TTS). Aujourd'hui, on regarde l'autre architecture, le **speech-to-speech**, puis on apprend à **mesurer** Awa et à la **mettre en production**.

| Section | Module | Statut |
|---|---|---|
| 0 | Configuration | disponible |
| 11 | L'architecture speech-to-speech | disponible |
| 12 | Évaluer un agent vocal | disponible |
| 13 | Production, sécurité et conformité | disponible |

**Fil rouge** : Awa, l'assistante commerciale vocale de Baobab Software (entreprise fictive).

**Niveaux d'exercices**
- **[Guidé]** : code à compléter aux endroits marqués `# TODO`, avec un indice.
- **[Avancé]** : consigne ouverte, pour aller plus loin.

## 0. Configuration

Exécutez cette cellule une fois au début de la journée. Le module 11 fonctionne **entièrement hors ligne** : la session Realtime est simulée. Une cellule **optionnelle** ouvre une vraie session si vous avez une clé OpenAI.

In [ ]:
# Installation (décommentez si nécessaire, par exemple sur Colab)
# !pip install -q numpy matplotlib

import os, re, time, json, asyncio, base64
import numpy as np
import matplotlib.pyplot as plt

NAVY, TEAL, AMBER, GRAY, LIGHT = "#19233E", "#1E8C95", "#F2A93B", "#C9CFDB", "#9CCFD3"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

CLE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print("Configuration prête.", "Clé OpenAI détectée." if CLE_OPENAI else "Mode hors ligne (aucune clé API détectée).")

---
## 11. L'architecture speech-to-speech : Awa sans texte au milieu

**Objectifs**
1. Comprendre ce que le **speech-to-speech** (S2S) change par rapport à la **cascade** STT, LLM, TTS : ce qu'il garde, ce qu'il fait perdre.
2. Savoir ce qu'est un **chunk audio** dans une session Realtime et calculer sa taille.
3. **Lire les événements** d'une session Realtime (`session.created`, `input_audio_buffer.speech_started`, `response.output_audio.delta`, etc.).
4. Comparer les deux modes de VAD d'OpenAI Realtime : **`server_vad`** et **`semantic_vad`**.
5. Gérer un **appel d'outil** en S2S (`check_availability`), côté OpenAI Realtime et côté Gemini Live (`NON_BLOCKING`, `INTERRUPT`, `WHEN_IDLE`, `SILENT`).
6. Peser le **compromis entre contrôle et naturel**.

**Durée** : 30 minutes d'exercices, puis l'atelier (20 minutes) et le quiz (10 minutes). Tout fonctionne **hors ligne** : la session Realtime est **simulée**. Une cellule optionnelle ouvre une vraie session si vous avez une clé OpenAI.

**Attention aux chiffres** : les délais, les budgets et les notes de ce notebook sont **fictifs** (repère indicatif). Les **noms** d'événements, de paramètres et de valeurs sont ceux de la documentation (OpenAI Realtime, Gemini Live) et des SDK, vérifiés le jour de l'écriture : l'API évolue vite, revérifiez-les le jour J.

In [ ]:
# Cellule de reprise du Module 11
import re, time, json, asyncio, base64, math
import numpy as np, matplotlib.pyplot as plt
try:
    NAVY
except NameError:
    raise RuntimeError("Exécutez d'abord la section 0 (Configuration).")
RED = "#C0392B"
print("Module 11 prêt.")

### 11.1 Cascade ou speech-to-speech ?

**En cascade** (jours 2 et 3), la parole du prospect devient du **texte** (STT), le LLM répond en **texte**, et le TTS le remet en **voix**. Le texte est un **goulot** : on le contrôle facilement (on peut le **lire**, le **corriger**, le **filtrer**), mais tout ce qui n'est pas dans le texte est **perdu** : l'intonation, l'hésitation, le rythme, l'hésitation entre deux mots.

**En speech-to-speech**, un **seul modèle** reçoit l'**audio** et produit l'**audio**. Il entend le ton, et peut répondre avec un rythme naturel, sans passer par du texte. En contrepartie, il est plus **difficile à contrôler** : pas de texte intermédiaire à filtrer avant la voix.

| | Cascade | Speech-to-speech |
|---|---|---|
| Ce que reçoit le LLM | du texte | de l'audio |
| Ce que produit le modèle | du texte, puis le TTS | de l'audio directement |
| Contrôle avant la voix | oui (filtre, normalisation) | limité |
| Choix des briques | libre (STT, LLM, TTS) | un seul modèle |
| Naturel (ton, rythme) | dépend du TTS | en général plus naturel (repère indicatif) |

Dans les deux cas, le **budget de latence** compte (module 8). Voici un budget **fictif** pour comparer : la cascade additionne quatre étapes après la fin de tour, le S2S en a une.

In [ ]:
BUDGET_CASCADE = {"fin de tour": 250, "STT final": 150, "LLM premier token": 300, "TTS premier octet": 200, "réseau et lecture": 150}
BUDGET_S2S = {"fin de tour": 250, "modèle audio, premier audio": 400, "réseau et lecture": 150}
print("Budgets FICTIFS (repère indicatif), en ms :")
print("  cascade :", BUDGET_CASCADE)
print("  s2s     :", BUDGET_S2S)

**[Guidé] Exercice 11.1** : écrivez `total_et_plus_gros_poste(budget)`. Elle renvoie le **total** en ms et le **nom du plus gros poste**.

*Indice* : `sum(budget.values())` pour le total, `max(budget, key=budget.get)` pour le poste.

In [ ]:
def total_et_plus_gros_poste(budget):
    # TODO : renvoyer (total, nom_du_plus_gros_poste)
    return (0, "")

for nom, budget, attendu in (("cascade", BUDGET_CASCADE, (1050, "LLM premier token")), ("s2s", BUDGET_S2S, (800, "modèle audio, premier audio"))):
    obtenu = total_et_plus_gros_poste(budget)
    print(("ok       " if obtenu == attendu else "À REVOIR  ") + f"{nom:8} -> {obtenu}")

Ne tirez **pas** de conclusion de ces chiffres : ils sont inventés. Dans le module 12, vous **mesurerez** la latence médiane et le p95 de vos deux Awa sur les mêmes scénarios, et c'est ce qui décidera.

### 11.2 L'audio dans une session Realtime

Un modèle de langage travaille sur des **tokens**. Un modèle audio de bout en bout travaille sur des tokens **audio** : le son est découpé et codé en unités discrètes, que le modèle lit et produit comme le texte. Vous n'avez pas à les fabriquer : l'API reçoit et renvoie de l'**audio brut**, et vous gérez des **chunks**.

D'après la documentation OpenAI Realtime, les exemples utilisent du **PCM à 24 kHz, mono**, envoyé en **base64** dans des événements `input_audio_buffer.append` (jusqu'à 15 Mo par chunk). Le PCM 16 bits prend **2 octets par échantillon** (module 2) ; le base64 écrit 3 octets avec 4 caractères.

**[Guidé] Exercice 11.2** : écrivez `taille_chunk(duree_ms, taux=24000)`. Elle renvoie un tuple `(octets_pcm, caracteres_base64)` pour un chunk de `duree_ms` millisecondes de PCM 16 bits mono.

*Indice* : nombre d'échantillons = `taux * duree_ms / 1000` ; chaque échantillon fait 2 octets ; le base64 d'un tableau de `n` octets fait `4 * ceil(n / 3)` caractères (`math.ceil`).

In [ ]:
def taille_chunk(duree_ms, taux=24000):
    # TODO : octets PCM puis longueur en base64
    return (0, 0)

TESTS_CHUNK = [(100, (4800, 6400)), (20, (960, 1280)), (1000, (48000, 64000))]
for duree, attendu in TESTS_CHUNK:
    obtenu = taille_chunk(duree)
    print(("ok       " if obtenu == attendu else "À REVOIR  ") + f"{duree:5} ms -> {obtenu}")

In [ ]:
# Vérification avec de vrais octets : 100 ms de silence PCM 16 bits à 24 kHz
pcm = np.zeros(2400, dtype="<i2").tobytes()
b64 = base64.b64encode(pcm).decode("ascii")
print(len(pcm), "octets PCM ->", len(b64), "caractères base64 (+33 %)")

### 11.3 Lire les événements d'une session Realtime

Une session Realtime est une **conversation d'événements** JSON entre votre code (le **client**) et le modèle (le **serveur**), sur WebSocket ou WebRTC. Les noms ci-dessous viennent de la documentation OpenAI et du SDK :

| Qui | Événement | Sens |
|---|---|---|
| serveur | `session.created` | la session est prête |
| client | `session.update` | je change la configuration (VAD, voix, instructions, outils) |
| serveur | `session.updated` | confirmation |
| serveur | `input_audio_buffer.speech_started` / `speech_stopped` | le VAD a vu le début / la fin de la parole |
| serveur | `input_audio_buffer.committed` | l'audio du tour est validé |
| serveur | `response.created` | le modèle commence à répondre |
| serveur | `response.output_audio.delta` | un morceau d'audio de la réponse |
| serveur | `response.output_audio_transcript.delta` | un morceau du texte de ce qui est dit |
| serveur | `response.function_call_arguments.done` | le modèle demande un outil |
| serveur | `response.done` | la réponse est terminée |

Une session dure **60 minutes au maximum**, et la **voix** ne peut plus changer après le premier audio produit (documentation).

On ne se connecte pas encore : on **rejoue** une session **simulée**, avec les vrais noms d'événements et des délais **fictifs**. Deux tours : une réponse directe, puis une demande de rendez-vous qui déclenche `check_availability`, avec un **préambule** (le modèle dit une phrase avant l'outil).

In [ ]:
def session_simulee(avec_preambule=True):
    """Événements serveur d'une session fictive : liste de dicts avec t (ms) et type."""
    e = []
    def ajouter(t, type_, **extra): e.append({"t": t, "type": type_, **extra})
    ajouter(0, "session.created")
    ajouter(20, "session.updated")
    # tour 1 : réponse directe
    ajouter(500, "input_audio_buffer.speech_started")
    ajouter(2600, "input_audio_buffer.speech_stopped")
    ajouter(2610, "input_audio_buffer.committed")
    ajouter(2640, "response.created")
    for k in range(5):
        ajouter(3010 + 100 * k, "response.output_audio.delta")
    ajouter(3600, "response.done")
    # tour 2 : outil check_availability
    ajouter(8000, "input_audio_buffer.speech_started")
    ajouter(9200, "input_audio_buffer.speech_stopped")
    ajouter(9210, "input_audio_buffer.committed")
    ajouter(9240, "response.created")
    if avec_preambule:
        for k in range(3):
            ajouter(9650 + 100 * k, "response.output_audio.delta")        # « Je regarde l'agenda. »
    ajouter(9800 if avec_preambule else 9500, "response.function_call_arguments.done", name="check_availability",
            call_id="call_1", arguments=json.dumps({"date": "2026-11-10"}))
    ajouter(9850 if avec_preambule else 9550, "response.done")
    # le client renvoie le résultat après 1500 ms d'outil lent, puis demande une réponse
    t0 = (9800 if avec_preambule else 9500) + 1500
    ajouter(t0 + 40, "response.created")
    for k in range(5):
        ajouter(t0 + 350 + 100 * k, "response.output_audio.delta")
    ajouter(t0 + 900, "response.done")
    return e

EVTS = session_simulee()
print(len(EVTS), "événements. Les dix premiers :")
for ev in EVTS[:10]:
    print(f"  {ev['t']:6} ms  {ev['type']}")

**[Guidé] Exercice 11.3** : écrivez `resumer_evenements(evenements)`. Elle renvoie un dictionnaire :
- `"tours"` : le nombre de `input_audio_buffer.speech_started` ;
- `"delais_premier_audio"` : pour chaque `speech_stopped`, le délai en ms jusqu'au **premier** `response.output_audio.delta` qui le suit ;
- `"outils"` : la liste des noms d'outils demandés (`response.function_call_arguments.done`).

*Indice* : parcourez la liste ; gardez en mémoire l'instant du dernier `speech_stopped` ; au premier `delta` qui suit, calculez la différence, puis oubliez l'instant (mettez-le à `None`) pour ne compter qu'**un** délai par tour.

In [ ]:
def resumer_evenements(evenements):
    tours, delais, outils = 0, [], []
    arret = None
    for ev in evenements:
        # TODO : compter les speech_started ; mémoriser l'instant du speech_stopped ;
        #        au premier output_audio.delta suivant, ajouter le délai ; noter les noms d'outils
        pass
    return {"tours": tours, "delais_premier_audio": delais, "outils": outils}

attendu = {"tours": 2, "delais_premier_audio": [410, 450], "outils": ["check_availability"]}
obtenu = resumer_evenements(EVTS)
print(("ok       " if obtenu == attendu else "À REVOIR  ") + str(obtenu))

In [ ]:
sans = resumer_evenements(session_simulee(avec_preambule=False))
print("Avec préambule  : délais jusqu'au premier audio", resumer_evenements(EVTS)["delais_premier_audio"], "ms")
print("Sans préambule  : délais jusqu'au premier audio", sans["delais_premier_audio"], "ms")

Le **préambule** change tout : avec lui, le prospect entend la voix d'Awa **450 ms** après avoir fini de parler, même si l'outil met 1,5 s à répondre. Sans lui, il attend plus de **2 s** (fictif, mais la mécanique est réelle). C'est le même principe que le filler du module 10. Côté OpenAI, la documentation de prompting indique que les modèles Realtime récents génèrent des préambules par défaut et recommande une **phrase courte** avant un outil, à ne pas dire pour un appel léger ; c'est donc une **instruction du prompt**, pas un réglage (les noms et les versions de modèles évoluent : testez).

### 11.4 `server_vad` contre `semantic_vad`

OpenAI Realtime propose deux détecteurs de **fin de tour** (champ `turn_detection` de la session) :

- **`server_vad`** : découpe l'audio selon le **silence**. Paramètres : `threshold` (seuil d'activation, de 0 à 1), `prefix_padding_ms` (audio gardé avant la parole détectée) et `silence_duration_ms` (silence qui signale la fin de la parole). L'exemple de la documentation utilise 0,5, 300 et 500.
- **`semantic_vad`** : regarde si les **mots** de l'utilisateur laissent penser qu'il a **fini**. Un seul paramètre de réglage : `eagerness` (`low` laisse l'utilisateur prendre son temps, `high` répond dès que possible, `auto` équivaut à `medium`).

Dans les deux cas, `create_response` (le modèle répond tout seul à la fin du tour) et `interrupt_response` (la parole de l'utilisateur coupe la réponse en cours) se règlent aussi. C'est exactement l'opposition du **module 9** : le silence seul (VAD) contre le **sens** (règle sémantique).

Voici la configuration d'une session avec `server_vad` (d'après l'exemple de la documentation) :

In [ ]:
SESSION_UPDATE = {
    "type": "session.update",
    "session": {
        "type": "realtime",
        "audio": {"input": {"turn_detection": {
            "type": "server_vad", "threshold": 0.5, "prefix_padding_ms": 300, "silence_duration_ms": 500,
            "create_response": True, "interrupt_response": True}}},
    },
}
SESSION_UPDATE_SEMANTIQUE = {
    "type": "session.update",
    "session": {"type": "realtime", "audio": {"input": {"turn_detection": {
        "type": "semantic_vad", "eagerness": "low", "create_response": True, "interrupt_response": True}}}},
}
print(json.dumps(SESSION_UPDATE["session"]["audio"]["input"]["turn_detection"], indent=2))
print(json.dumps(SESSION_UPDATE_SEMANTIQUE["session"]["audio"]["input"]["turn_detection"], indent=2))

**Un modèle jouet pour comparer.** La documentation ne donne **pas** les délais exacts de `eagerness`. On utilise donc des valeurs **fictives** : pour chaque niveau, un délai si la phrase est **complète** et un délai plus long si elle est **inachevée**. Le but est de voir la **forme** du compromis, pas de prédire un vrai comportement.

Chaque énoncé est décrit par son texte, sa **pause** (en ms, avant que le prospect reprenne s'il n'a pas fini, ou avant que le tour ne se termine), et s'il est **complet**.

In [ ]:
ENONCES = [
    ("Oui, mardi à dix heures.", 700, True),
    ("Je voudrais un rendez-vous pour", 900, False),
    ("Mon numéro c'est le 77 616", 1200, False),
    ("Nous sommes huit salariés", 600, True),
    ("Je ne sais pas si", 1500, False),
    ("Quelles sont vos offres ?", 500, True),
    ("Mon budget c'est environ", 1000, False),
    ("Non merci", 800, True),
]
EAGERNESS = {"high": (400, 900), "medium": (600, 1300), "low": (800, 2000)}   # FICTIF : (si complet, si inachevé), en ms
print(len(ENONCES), "énoncés,", sum(1 for e in ENONCES if not e[2]), "inachevés")

**[Guidé] Exercice 11.4** : écrivez `simuler(config, enonces)`. Elle renvoie `(coupures, attente_moyenne)`.
- `config` est `{"type": "server_vad", "silence_ms": 500}` ou `{"type": "semantic_vad", "eagerness": "medium"}`.
- Pour un énoncé **inachevé**, le prospect reprend après sa `pause` : si le délai d'attente de la config est **inférieur ou égal** à la pause, il est **coupé** (une coupure).
- Pour un énoncé **complet**, il n'y a pas de coupure ; le délai d'attente de la config s'ajoute à la réponse.
- `server_vad` attend toujours `silence_ms`. `semantic_vad` attend le délai **complet** ou **inachevé** de `EAGERNESS`, selon l'énoncé.
- `attente_moyenne` est la **moyenne des attentes sur les énoncés complets**.

*Indice* : écrivez une petite fonction interne `delai(complet)` qui renvoie le délai selon la config.

In [ ]:
def simuler(config, enonces):
    coupures, attentes = 0, []
    # TODO : pour chaque (texte, pause, complet) : calculer le délai de la config ;
    #        inachevé et délai <= pause -> coupure ; complet -> ajouter le délai aux attentes
    return coupures, (sum(attentes) / len(attentes) if attentes else 0)

TESTS_VAD = [
    ({"type": "server_vad", "silence_ms": 500}, (4, 500)),
    ({"type": "server_vad", "silence_ms": 1000}, (3, 1000)),
    ({"type": "semantic_vad", "eagerness": "high"}, (4, 400)),
    ({"type": "semantic_vad", "eagerness": "medium"}, (1, 600)),
    ({"type": "semantic_vad", "eagerness": "low"}, (0, 800)),
]
bons = 0
for config, attendu in TESTS_VAD:
    obtenu = simuler(config, ENONCES)
    ok = obtenu == attendu
    bons += ok
    print(("ok       " if ok else "À REVOIR  ") + f"{str(config):55} -> {obtenu}")
print(f"\n{bons}/{len(TESTS_VAD)} corrects")

In [ ]:
CONFIGS = [("server 500 ms", {"type": "server_vad", "silence_ms": 500}), ("server 1000 ms", {"type": "server_vad", "silence_ms": 1000}),
           ("sémantique high", {"type": "semantic_vad", "eagerness": "high"}), ("sémantique medium", {"type": "semantic_vad", "eagerness": "medium"}),
           ("sémantique low", {"type": "semantic_vad", "eagerness": "low"})]
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for i, (nom, cfg) in enumerate(CONFIGS):
    c, a = simuler(cfg, ENONCES)
    ax.scatter(a, c, s=90, color=NAVY if cfg["type"] == "server_vad" else TEAL, zorder=3)
    ax.annotate(nom, (a, c), textcoords="offset points", xytext=(8, 6), fontsize=9)
ax.set_xlabel("attente moyenne sur les énoncés complets (ms)"); ax.set_ylabel("énoncés inachevés coupés (sur 4)")
ax.set_title("Modèle jouet FICTIF : attente contre coupures", fontsize=10, color="#5E6779")
ax.set_ylim(-0.5, 4.6); plt.tight_layout(); plt.show()

**Lire le graphique.** Un **silence fixe** (`server_vad`) force un choix : attente courte et beaucoup de coupures, ou peu de coupures et une attente longue. Le détecteur **sémantique** peut rester **court** sur les phrases complètes et **patient** sur les inachevées : c'est son intérêt. Mais ce n'est **pas gratuit** : il décide à votre place, avec un modèle que vous ne réglez qu'avec `eagerness`. Et ces valeurs sont fictives ; sur vos enregistrements, l'écart peut être très différent (module 12).

### 11.5 Les outils en speech-to-speech

Awa garde ses outils (`check_availability`, `create_lead`, `book_meeting`, `transfer_to_human`), mais le **protocole** change.

**OpenAI Realtime.** Vous déclarez les outils dans la session. Quand le modèle en veut un, le serveur envoie `response.function_call_arguments.done` avec le `name`, les `arguments` (du JSON **en texte**) et un `call_id`. Vous exécutez l'outil **côté serveur**, puis vous envoyez **deux** événements : un `conversation.item.create` contenant un item `function_call_output` (avec le **même** `call_id` et la sortie en JSON texte), puis un `response.create` pour que le modèle **parle** (noms vérifiés dans les types du SDK `openai`).

**Gemini Live.** Par défaut, les appels de fonction sont **bloquants** : la conversation s'arrête tant que le résultat n'est pas là. On ajoute `"behavior": "NON_BLOCKING"` à la déclaration de la fonction pour que le modèle **continue de parler** pendant l'exécution. En renvoyant le résultat (`FunctionResponse`), on choisit un `scheduling` :

| `scheduling` | Effet (documentation Gemini Live) |
|---|---|
| `INTERRUPT` | interrompt la tâche en cours et informe le modèle tout de suite |
| `WHEN_IDLE` | attend que la tâche en cours soit terminée |
| `SILENT` | ne fait rien tout de suite, le résultat servira plus tard dans la conversation |

La page de la documentation indique que l'appel **asynchrone** est pris en charge sur Gemini 2.5 Flash Live (aperçu) mais pas sur Gemini 3.1 Flash Live (aperçu), où le modèle ne répond qu'après l'envoi de la réponse de l'outil. Les exemples de la page citent un nom de modèle qui ne correspond pas à ce tableau : **revérifiez le nom du modèle** avant de coder.

**[Guidé] Exercice 11.5** : écrivez `reponse_outil_openai(evt, resultat)`. Elle reçoit l'événement `response.function_call_arguments.done` et le **résultat** (un dictionnaire), et renvoie la **liste des deux événements** à envoyer :
1. `{"type": "conversation.item.create", "item": {"type": "function_call_output", "call_id": ..., "output": ...}}` où `output` est le résultat **en JSON texte** (`json.dumps`) ;
2. `{"type": "response.create"}`.

*Indice* : le `call_id` se lit dans `evt["call_id"]`.

In [ ]:
AGENDA_SIMULE = {"2026-11-10": ["09:00", "09:30", "14:30"], "2026-11-11": []}

def check_availability(date):
    return {"date": date, "creneaux": AGENDA_SIMULE.get(date, [])}

EVT_OUTIL = {"type": "response.function_call_arguments.done", "call_id": "call_1", "name": "check_availability",
             "arguments": json.dumps({"date": "2026-11-10"})}

def reponse_outil_openai(evt, resultat):
    # TODO : renvoyer la liste [function_call_output, response.create]
    return []

evts = reponse_outil_openai(EVT_OUTIL, check_availability("2026-11-10"))
ok = (len(evts) == 2 and evts[0].get("type") == "conversation.item.create"
      and evts[0].get("item", {}).get("type") == "function_call_output"
      and evts[0]["item"].get("call_id") == "call_1"
      and json.loads(evts[0]["item"].get("output", "null")) == {"date": "2026-11-10", "creneaux": ["09:00", "09:30", "14:30"]}
      and evts[1] == {"type": "response.create"})
print(("ok       " if ok else "À REVOIR  ") + "deux événements à renvoyer : " + str([e.get("type") for e in evts]))

**[Avancé] Exercice 11.6** : `check_availability` en speech-to-speech, pour **les deux** fournisseurs. Écrivez `traiter_appel_outil(evt, archi, situation)` :

- `archi == "openai"` : `evt` est un événement `response.function_call_arguments.done` (arguments en **JSON texte**). Renvoyez la liste des **deux événements** de l'exercice 11.5, avec le résultat de `check_availability`.
- `archi == "gemini"` : `evt` est de la forme `{"id": "fc_1", "name": "check_availability", "args": {"date": "2026-11-10"}}`. Renvoyez **un dictionnaire** `{"id": ..., "name": ..., "response": {...résultat..., "scheduling": X}}` où `X` est choisi par la règle ci-dessous (forme du `FunctionResponse` de la documentation : `response={"result": ..., "scheduling": ...}`).
- Un **outil inconnu** renvoie un résultat `{"erreur": "outil inconnu"}`, sans `scheduling` pour OpenAI.

**Règle de scheduling pour Awa** (`situation` est un dictionnaire) :
- `transfer_to_human` : `"INTERRUPT"` (on ne fait pas attendre un client en colère) ;
- `create_lead` : `"SILENT"` (inutile d'en parler au prospect) ;
- `check_availability` et `book_meeting` : `"WHEN_IDLE"` si le prospect attend (`situation["prospect_attend"]` vrai), sinon `"SILENT"`.

Les tests plus bas couvrent les deux fournisseurs.

In [ ]:
def choisir_scheduling(outil, situation):
    # TODO : appliquer la règle de scheduling
    return "WHEN_IDLE"

def traiter_appel_outil(evt, archi, situation):
    # TODO : exécuter l'outil, puis formater la réponse selon l'architecture
    return None

def tester_outils_s2s():
    echecs = []
    s = {"prospect_attend": True}
    if choisir_scheduling("check_availability", s) != "WHEN_IDLE": echecs.append("check_availability en attente : WHEN_IDLE")
    if choisir_scheduling("check_availability", {"prospect_attend": False}) != "SILENT": echecs.append("sans attente : SILENT")
    if choisir_scheduling("create_lead", s) != "SILENT": echecs.append("create_lead : SILENT")
    if choisir_scheduling("transfer_to_human", s) != "INTERRUPT": echecs.append("transfer_to_human : INTERRUPT")
    r = traiter_appel_outil(EVT_OUTIL, "openai", s)
    if not (isinstance(r, list) and len(r) == 2 and r[0]["item"]["call_id"] == "call_1" and r[1] == {"type": "response.create"}):
        echecs.append("openai : deux événements")
    elif json.loads(r[0]["item"]["output"])["creneaux"] != ["09:00", "09:30", "14:30"]:
        echecs.append("openai : créneaux de l'agenda")
    g = traiter_appel_outil({"id": "fc_1", "name": "check_availability", "args": {"date": "2026-11-10"}}, "gemini", s)
    if not (isinstance(g, dict) and g.get("id") == "fc_1" and g.get("name") == "check_availability"
            and g.get("response", {}).get("scheduling") == "WHEN_IDLE" and g["response"].get("creneaux") == ["09:00", "09:30", "14:30"]):
        echecs.append("gemini : id, name, response avec scheduling")
    inconnu = traiter_appel_outil({"id": "fc_2", "name": "meteo", "args": {}}, "gemini", s)
    if not (isinstance(inconnu, dict) and inconnu.get("response", {}).get("erreur") == "outil inconnu"):
        echecs.append("outil inconnu : erreur")
    print("Tous les tests passent." if not echecs else "À REVOIR : " + " ; ".join(echecs))

tester_outils_s2s()

**Ce que ça change pour Awa.** En cascade, `check_availability` passait par `avec_attente` (module 10) : un filler par `session.say`, qui ne passe pas par le LLM. En S2S, **il n'y a plus de TTS à part** : le filler est un **préambule** que le **modèle** dit lui-même, selon vos **instructions**. Et pour un outil long, Gemini Live offre `NON_BLOCKING` : le modèle continue de parler pendant que l'agenda répond. Le **timeout** reste **votre** travail dans les deux cas.

### 11.6 Contrôle contre naturel

Toutes vos protections des modules 7 et 10 reposaient sur le **texte** :
- la **normalisation** avant le TTS (« 14 h 30 », « 49 000 francs ») ;
- le **filtre** sur la sortie (pas de remise inventée) ;
- le **lexique** de post-correction du STT (module 5).

En S2S, il n'y a plus de texte **avant** la voix. Vous avez le **prompt**, les **outils**, et le **transcript** de ce qui a été dit **après coup** (il sert à journaliser et à évaluer, pas à bloquer). Vous perdez aussi le **choix de la voix** (celles du fournisseur, par exemple `marin` ou `cedar` chez OpenAI) et la lecture **contrôlée** des numéros. À l'inverse, le **ton**, les **hésitations** et le **rythme** sont plus naturels, en général (repère indicatif : c'est à **entendre**).

Pour décider, il faut **pondérer**. Voici des notes de départ **fictives** (de 1 à 5), à **contester** pendant l'atelier :

In [ ]:
CRITERES = ["naturel", "contrôle du texte", "latence", "coût maîtrisé", "choix des briques"]
NOTES = {"cascade": [3, 5, 3, 4, 5], "s2s": [5, 2, 4, 3, 2]}      # FICTIF : jugement de départ
print(f"{'critère':20}" + "".join(f"{a:>10}" for a in NOTES))
for i, c in enumerate(CRITERES):
    print(f"{c:20}" + "".join(f"{NOTES[a][i]:>10}" for a in NOTES))

**[Guidé] Exercice 11.7** : écrivez `score(notes, poids)`, la **somme pondérée** des notes (poids de même longueur, qui **somment à 1**). Puis `choisir_architecture(poids)` qui renvoie le **nom** de l'architecture au **meilleur** score.

*Indice* : `sum(n * p for n, p in zip(notes, poids))`. En cas d'**égalité**, renvoyez `"cascade"` (plus facile à contrôler).

In [ ]:
def score(notes, poids):
    # TODO : somme pondérée
    return 0

def choisir_architecture(poids):
    # TODO : comparer les scores de NOTES["cascade"] et NOTES["s2s"] ; égalité -> "cascade"
    return "cascade"

PROFILS_POIDS = {
    "vente, contrôle d'abord": [0.15, 0.40, 0.15, 0.15, 0.15],
    "accueil, naturel d'abord": [0.40, 0.10, 0.25, 0.15, 0.10],
    "poids égaux": [0.2, 0.2, 0.2, 0.2, 0.2],
}
ATTENDU_ARCHI = {"vente, contrôle d'abord": "cascade", "accueil, naturel d'abord": "s2s", "poids égaux": "cascade"}
for nom, p in PROFILS_POIDS.items():
    r = choisir_architecture(p)
    print(("ok       " if r == ATTENDU_ARCHI[nom] else "À REVOIR  ") + f"{nom:28} -> {r}")

Le choix **change avec les poids** : c'est pour cela qu'on ne choisit pas une architecture sur une démonstration, mais sur des **chiffres** mesurés avec **vos** priorités. Au module 12, vous mesurerez le taux de rendez-vous, l'exactitude des données du CRM, la latence médiane et p95, le WER sur les noms et numéros, et le coût par appel, **pour les deux Awa**.

### 11.7 Ouvrir une vraie session Realtime (optionnel)

Si vous avez une clé OpenAI, cette cellule ouvre une **vraie** session, envoie la configuration du 11.4, demande une réponse en **texte** à « Bonjour » (pour ne pas avoir à gérer l'audio) et **liste les types d'événements** reçus. Sans clé, elle ne fait rien.

*Non testée avec une vraie clé au moment de l'écriture : elle utilise `client.realtime.connect(...)` du SDK `openai` et les événements ci-dessus. En cas d'écart, la documentation OpenAI Realtime a raison.*

In [ ]:
async def session_reelle(modele="gpt-realtime", duree_max_s=20):
    if not CLE_OPENAI:
        print("Pas de clé OpenAI : étape optionnelle ignorée.")
        return None
    from openai import AsyncOpenAI
    client = AsyncOpenAI()
    types_recus = []
    async with client.realtime.connect(model=modele) as conn:
        await conn.send(SESSION_UPDATE)
        await conn.send({"type": "conversation.item.create",
                         "item": {"type": "message", "role": "user", "content": [{"type": "input_text", "text": "Bonjour"}]}})
        await conn.send({"type": "response.create", "response": {"output_modalities": ["text"]}})
        fin = time.time() + duree_max_s
        while time.time() < fin:
            evt = await asyncio.wait_for(conn.recv(), timeout=duree_max_s)
            types_recus.append(evt.type)
            if evt.type == "response.done":
                break
    print(types_recus)
    return types_recus

await session_reelle()

### 11.8 Atelier : Awa en speech-to-speech contre Awa en cascade

Le dépôt contient `awa_agent.py` : la variable d'environnement `AWA_ARCHI` choisit l'architecture (`cascade` par défaut, `realtime` pour le speech-to-speech). **Les outils, le CRM simulé et le prompt sont les mêmes** ; seule la façon de parler change.

**À faire en binôme** (avec des clés, sinon sur le papier) :
1. Lancez chaque Awa en console (`python awa_agent.py console`), sur **les mêmes quatre scénarios** : Mamadou qui réserve un rendez-vous, Aïssatou qui demande le tarif Entreprise, Jean-Pierre qui veut un humain, Khady (hors cible).
2. Notez pour **chaque** Awa : le délai avant la première parole (à l'oreille), la façon de dire les **numéros** et les **montants**, la réaction à une **interruption**, et si Awa **invente** une remise.
3. Complétez le tableau ci-dessous : une note de 1 à 5 par critère, **vos** notes (et pas celles du 11.6), puis calculez le résultat avec vos poids.

In [ ]:
MES_NOTES = {"cascade": [0, 0, 0, 0, 0], "s2s": [0, 0, 0, 0, 0]}      # TODO : vos notes, de 1 à 5, dans l'ordre des CRITERES
MES_POIDS = [0.2, 0.2, 0.2, 0.2, 0.2]                                  # TODO : vos priorités (somme = 1)

def bilan_atelier(notes, poids):
    for archi, n in notes.items():
        print(f"{archi:8} -> {score(n, poids):.2f}")
bilan_atelier(MES_NOTES, MES_POIDS)

**Questions pour l'atelier** (à répondre dans une cellule de texte) :
1. Quels critères avez-vous **pondérés le plus**, et **pourquoi** pour une assistante commerciale ?
2. Quelles **protections** de la cascade (normalisation, filtre de remise, lexique) vous manquent le plus en S2S, et comment les **remplacer** (prompt, outils, transcript) ?
3. Que **ne pouvez-vous pas mesurer** sans le module 12 ?
4. Dans quel cas **précis** garderiez-vous la cascade, et dans quel cas passeriez-vous au S2S ?

### À retenir

- La **cascade** passe par du **texte** : facile à contrôler, mais le ton, les hésitations et le rythme sont perdus. Le **speech-to-speech** garde l'audio de bout en bout : plus naturel, plus difficile à contrôler.
- Une session **Realtime** est une conversation d'**événements** JSON ; l'audio y circule en **chunks** (PCM 24 kHz mono, en base64 sur WebSocket).
- **`server_vad`** décide au **silence** (`threshold`, `prefix_padding_ms`, `silence_duration_ms`) ; **`semantic_vad`** décide d'après les **mots** (`eagerness`).
- Un **outil** en Realtime : `response.function_call_arguments.done`, puis `function_call_output` avec le **même** `call_id`, puis `response.create`. Le **préambule** est une **instruction** au modèle.
- Chez Gemini Live, `NON_BLOCKING` laisse parler le modèle pendant l'outil ; `INTERRUPT`, `WHEN_IDLE` et `SILENT` règlent **quand** le résultat est dit.
- En S2S, les protections fondées sur le **texte avant la voix** (normalisation, filtre) disparaissent : on garde le **prompt**, les **outils** et le **transcript** après coup. On choisit sur des **chiffres** (module 12).

---
### Quiz du Module 11

**Q1.** Que perd-on dans une cascade STT, LLM, TTS qu'un modèle speech-to-speech peut conserver ?
a) le texte brut de la conversation · b) la vitesse de lecture · c) l'intonation, les hésitations et le rythme, qui disparaissent quand l'audio devient du texte · d) le volume de l'appel

**Q2.** Le modèle OpenAI Realtime demande un outil. Comment lui renvoyer le résultat ?
a) un `conversation.item.create` avec un item `function_call_output` (même `call_id`), puis un `response.create` · b) un seul `response.create` sans le résultat · c) en modifiant le prompt avec `session.update` · d) en fermant puis rouvrant la session

**Q3.** Quelle est la différence entre `server_vad` et `semantic_vad` ?
a) `semantic_vad` ne sert qu'à la transcription · b) `server_vad` comprend le sens des mots · c) il n'y en a aucune, ce sont deux noms du même détecteur · d) `server_vad` décide de la fin du tour d'après le silence, `semantic_vad` d'après le sens des mots, avec `eagerness`

**Q4.** Chez Gemini Live, que fait `scheduling: "SILENT"` ?
a) il coupe le micro du prospect · b) il n'agit pas tout de suite : le résultat de l'outil servira plus tard dans la conversation · c) il interrompt la réponse en cours · d) il annule l'appel de fonction

**Q5.** Quel avantage garde la cascade sur le speech-to-speech pour Awa ?
a) elle est toujours plus rapide · b) elle n'a pas besoin d'outils · c) on peut lire, filtrer et normaliser le texte avant qu'il ne soit prononcé · d) elle ne nécessite aucune clé API

In [ ]:
mes_reponses_m11 = {1: "", 2: "", 3: "", 4: "", 5: ""}   # TODO : vos réponses

In [ ]:
def corriger_quiz_module11(reponses):
    attendu = {1: "c", 2: "a", 3: "d", 4: "b", 5: "c"}
    explications = {
        1: "Le texte est un goulot : ton, hésitations et rythme n'y sont pas. Un modèle audio de bout en bout les entend.",
        2: "On renvoie un function_call_output avec le même call_id, puis on demande la réponse avec response.create.",
        3: "server_vad se fonde sur le silence (threshold, prefix_padding_ms, silence_duration_ms) ; semantic_vad sur le sens des mots (eagerness).",
        4: "SILENT : pas d'action immédiate, le résultat est utilisé plus tard. INTERRUPT interrompt, WHEN_IDLE attend la fin de la tâche en cours.",
        5: "En cascade, il y a du texte avant la voix : on peut le normaliser (module 7) et le filtrer (module 10). En S2S, il n'y en a pas.",
    }
    score_quiz = 0
    for q in attendu:
        r = str(reponses.get(q, "")).strip().lower()
        ok = r == attendu[q]; score_quiz += ok
        print(f"Q{q} : {'correct' if ok else 'à revoir'} (votre réponse : {r or '-'}). {explications[q]}")
    print(f"\nScore : {score_quiz}/5")

corriger_quiz_module11(mes_reponses_m11)

---
## 12. Évaluer un agent vocal : mesurer avant de choisir

**Objectifs**
1. Définir ce que veut dire **« Awa marche »** avec des **métriques** mesurables.
2. Construire un **jeu de 20 scénarios** couvrant les personas du fil rouge.
3. Calculer le **taux de rendez-vous**, l'**exactitude des données CRM**, la **latence médiane et p95**, le **WER sur les noms et numéros** et le **coût par appel**.
4. Lire un **tableau de bord** en tenant compte de l'**incertitude** (20 scénarios, c'est peu).
5. **[Avancé]** Utiliser un **LLM juge** avec une rubrique, et mesurer son **accord** avec un humain.
6. **Choisir entre cascade et speech-to-speech** à partir des chiffres, avec des **exigences** écrites à l'avance.

**Durée** : 30 minutes d'exercices, puis l'atelier (20 minutes) et le quiz (10 minutes). Tout fonctionne **hors ligne** : les appels sont **simulés**. Une cellule optionnelle appelle un vrai LLM juge si vous avez une clé OpenAI.

**Attention aux chiffres** : les **traces** de ce notebook sont **générées par un simulateur** avec des profils d'erreur **fictifs** pour la cascade et le speech-to-speech. **Aucun résultat ne dit laquelle des deux architectures est meilleure dans la vraie vie.** Ce qui est réel, c'est la **méthode** : c'est elle que vous appliquerez à vos vraies traces.

In [ ]:
# Cellule de reprise du Module 12
import re, time, json, math, random
import numpy as np, matplotlib.pyplot as plt
try:
    NAVY
except NameError:
    raise RuntimeError("Exécutez d'abord la section 0 (Configuration).")
RED = "#C0392B"
print("Module 12 prêt.")

### 12.1 La méthode : cinq étapes

1. **Définir le succès** avant de mesurer : « Awa marche » veut dire quoi, pour Baobab ?
2. **Écrire un jeu de scénarios** : les cas qu'on veut réussir, y compris les difficiles (accent, objection, colère, hors cible).
3. **Lancer** chaque scénario sur chaque agent (appels simulés, puis réels) et **enregistrer une trace** par appel.
4. **Calculer les métriques** sur les traces.
5. **Décider** avec des **exigences écrites à l'avance**, puis **recommencer** après chaque changement.

Les **métriques** du programme :

| Métrique | Question | Source |
|---|---|---|
| Taux de rendez-vous | Awa réserve-t-elle quand il le faut ? | trace de `book_meeting` |
| Exactitude des données CRM | La fiche contient-elle les bonnes valeurs ? | trace de `create_lead` |
| Latence médiane et p95 | Combien attend-on, **typiquement** et **dans les mauvais cas** ? | métriques par tour (module 8) |
| WER sur les noms et numéros | L'oreille comprend-elle les **entités** ? | transcription contre référence (modules 5 et 10) |
| Coût par appel | Combien ça coûte ? | durée de l'appel et tarifs |

On y ajoute une métrique de **conformité** (Awa n'invente pas de remise, module 10), qui est un **plancher** : un seul manquement compte.

### 12.2 Le jeu de 20 scénarios

Un bon jeu de scénarios **ressemble à vos appels réels**, avec les cas difficiles **sur-représentés** (on veut les voir). Voici 20 scénarios autour des personas du fil rouge :

| Type | Nombre | Attendu |
|---|---|---|
| `rdv` : Mamadou (8 salariés) et Aïssatou (120 salariés) | 10 | qualifier et réserver un rendez-vous |
| `objection` : « c'est cher », « envoyez-moi un e-mail », « je réfléchis » | 3 | répondre sans remise inventée ; réserver si le prospect accepte |
| `accent` : noms et numéros difficiles | 3 | noter correctement le nom et le numéro, puis réserver |
| `colere` : Jean-Pierre | 2 | transférer vers un humain, sans rendez-vous |
| `hors_cible` : Khady | 2 | conclure poliment, ni fiche ni rendez-vous |

In [ ]:
PRENOMS = ["Mamadou Ba", "Aïssatou Diallo", "Fatou Sow", "Ibrahima Ndiaye", "Khady Fall", "Moussa Sarr", "Awa Gueye", "Cheikh Diop",
           "Mariama Sy", "Ousmane Faye", "Ndeye Thiam", "Abdoulaye Kane", "Rokhaya Mbaye", "Modou Seck", "Coumba Ndao", "Alioune Ba",
           "Bineta Cissé", "Pape Sene", "Seynabou Diouf", "Lamine Toure"]
TELS = ["77 616 06 16", "78 245 31 09", "76 902 18 45", "70 334 27 61", "77 508 94 12", "78 119 63 70", "76 480 25 98", "70 871 42 36",
        "77 223 56 08", "78 640 71 29", "76 357 80 14", "70 995 12 63", "77 164 38 52", "78 723 49 07", "76 541 20 86", "70 288 65 31",
        "77 809 14 73", "78 076 52 40", "76 618 93 25", "70 453 07 18"]
TYPES = ["rdv"] * 10 + ["objection"] * 3 + ["accent"] * 3 + ["colere"] * 2 + ["hors_cible"] * 2
DELAIS = ["immediat", "1_mois", "3_mois", "plus_tard"]

def construire_scenarios():
    scenarios = []
    for i, type_ in enumerate(TYPES):
        crm = None if type_ in ("colere", "hors_cible") else {
            "nom": PRENOMS[i], "taille_entreprise": [8, 120, 15, 40, 6, 25, 10, 60, 12, 30][i % 10],
            "budget_mensuel": [25000, 49000, 49000, 25000, 25000, 49000, 25000, 49000, 25000, 49000][i % 10], "delai": DELAIS[i % 4]}
        scenarios.append({"id": i + 1, "type": type_, "persona": {"rdv": "Mamadou/Aïssatou", "objection": "prospect hésitant", "accent": "prospect à noms difficiles",
                          "colere": "Jean-Pierre", "hors_cible": "Khady"}[type_],
                          "rdv_attendu": type_ in ("rdv", "objection", "accent"), "crm_attendu": crm,
                          "transfert_attendu": type_ == "colere", "nom": PRENOMS[i], "tel": TELS[i],
                          "tours": {"rdv": 8, "objection": 9, "accent": 10, "colere": 5, "hors_cible": 4}[type_]})
    return scenarios

SCENARIOS = construire_scenarios()
from collections import Counter
print(len(SCENARIOS), "scénarios :", dict(Counter(s["type"] for s in SCENARIOS)))
print("Rendez-vous attendus :", sum(s["rdv_attendu"] for s in SCENARIOS), "| transferts attendus :", sum(s["transfert_attendu"] for s in SCENARIOS))
print("Exemple :", SCENARIOS[0])

### 12.3 Simuler des appels et enregistrer des traces

Une **trace** est l'enregistrement d'**un appel** : ce qui a été dit, les appels d'outils, les durées. Sans traces, pas d'évaluation. Dans la vraie vie, vous les obtenez de `awa_agent.py` (journal du module 8, `awa_metriques.jsonl`) et du CRM simulé.

Ici, un **simulateur** produit des traces pour deux agents : `cascade` et `s2s`, chacun avec un **profil d'erreur fictif**. Rien n'est calibré sur de vrais systèmes.

In [ ]:
PROFILS_SIM = {
    "cascade": {"lat_med": 1050, "sigma": 0.30, "p_pic": 0.05, "pic_ms": 2600,
                "p_rdv": {"rdv": 0.92, "objection": 0.72, "accent": 0.78}, "p_champ": 0.97,
                "p_mot": {"normal": 0.04, "accent": 0.09}, "p_transfert": 0.95, "p_remise": 0.00},
    "s2s": {"lat_med": 800, "sigma": 0.30, "p_pic": 0.04, "pic_ms": 2200,
            "p_rdv": {"rdv": 0.90, "objection": 0.78, "accent": 0.70}, "p_champ": 0.93,
            "p_mot": {"normal": 0.05, "accent": 0.11}, "p_transfert": 0.85, "p_remise": 0.10},
}

def _degrader(ref, p, rng):
    sortie = []
    for mot in ref.split():
        r = rng.random()
        if r < p / 3:
            continue
        if r < p:
            mot = str((int(mot) + rng.choice([1, 2, 3, 7])) % (10 ** len(mot))).zfill(len(mot)) if mot.isdigit() else mot + "s"
        sortie.append(mot)
    return " ".join(sortie)

def simuler_traces(archi, scenarios, graine=11):
    prof, rng = PROFILS_SIM[archi], random.Random(graine + (0 if archi == "cascade" else 1000))
    traces = []
    for sc in scenarios:
        lat = []
        for _ in range(sc["tours"]):
            v = prof["lat_med"] * math.exp(rng.gauss(0, prof["sigma"]))
            if rng.random() < prof["p_pic"]:
                v += prof["pic_ms"]
            lat.append(round(v))
        reserve = sc["rdv_attendu"] and rng.random() < prof["p_rdv"].get(sc["type"], 0)
        crm_saisi = None
        if sc["crm_attendu"] is not None and (reserve or rng.random() < 0.6):
            crm_saisi = {k: (v if rng.random() < prof["p_champ"] else (v + 1 if isinstance(v, int) else "erreur")) for k, v in sc["crm_attendu"].items()}
        ref_id = f"je m'appelle {sc['nom'].lower()} et mon numéro est le {sc['tel']}"
        p_mot = prof["p_mot"]["accent" if sc["type"] == "accent" else "normal"]
        transfert = sc["transfert_attendu"] and rng.random() < prof["p_transfert"]
        remise = sc["type"] == "objection" and rng.random() < prof["p_remise"] * 3
        traces.append({"scenario": sc["id"], "archi": archi, "type": sc["type"], "rdv_attendu": sc["rdv_attendu"], "rdv_reserve": bool(reserve),
                       "crm_attendu": sc["crm_attendu"], "crm_saisi": crm_saisi, "latences_ms": lat, "ref_id": ref_id,
                       "hyp_id": _degrader(ref_id, p_mot, rng), "tel": sc["tel"], "transfert_attendu": sc["transfert_attendu"],
                       "transfert_fait": bool(transfert), "minutes": round(sc["tours"] * 0.35 + rng.uniform(0, 1), 2),
                       "remise_inventee": bool(remise)})
    return traces

TRACES = {a: simuler_traces(a, SCENARIOS) for a in PROFILS_SIM}
print({a: len(t) for a, t in TRACES.items()}, "traces. Exemple de trace (cascade, scénario 1) :")
print(json.dumps({k: v for k, v in TRACES["cascade"][0].items() if k in ("scenario", "rdv_attendu", "rdv_reserve", "crm_saisi", "latences_ms", "hyp_id")}, ensure_ascii=False))

### 12.4 Le taux de rendez-vous

**Taux de rendez-vous** = (appels où un rendez-vous était **attendu** et a été **réservé**) / (appels où un rendez-vous était **attendu**).

On ne compte **pas** les scénarios où aucun rendez-vous n'était attendu (Jean-Pierre, Khady) : réserver à Khady **n'est pas** un succès.

**[Guidé] Exercice 12.1** : écrivez `taux_rdv(traces)`.

*Indice* : gardez les traces avec `rdv_attendu` vrai, puis calculez la part de celles où `rdv_reserve` est vrai. Si aucune trace n'attend de rendez-vous, renvoyez `0.0`.

In [ ]:
def taux_rdv(traces):
    # TODO : part des traces avec rdv_attendu où rdv_reserve est vrai
    return 0.0

FIXTURE_RDV = [
    {"rdv_attendu": True, "rdv_reserve": True}, {"rdv_attendu": True, "rdv_reserve": True},
    {"rdv_attendu": True, "rdv_reserve": False}, {"rdv_attendu": False, "rdv_reserve": False},
]
r = taux_rdv(FIXTURE_RDV)
print(("ok       " if abs(r - 2 / 3) < 1e-9 else "À REVOIR  ") + f"taux_rdv de la fixture : {r:.3f} (attendu 0.667)")

### 12.5 L'exactitude des données CRM

Un rendez-vous réservé avec une **fiche fausse** est un échec en différé : le commercial appelle le mauvais numéro, ou le mauvais budget. On mesure l'**exactitude champ par champ** sur les 4 champs de `create_lead` (nom, taille, budget, délai).

**Exactitude** = (champs **corrects**) / (champs **attendus**), sur les traces où une fiche était attendue (`crm_attendu` non vide). Si aucune fiche n'a été créée (`crm_saisi` vide), **tous** les champs attendus comptent comme faux.

**[Guidé] Exercice 12.2** : écrivez `exactitude_crm(traces)`.

*Indice* : pour chaque trace avec `crm_attendu`, parcourez ses champs ; un champ est correct si `crm_saisi` existe et contient la **même** valeur.

In [ ]:
def exactitude_crm(traces):
    # TODO : champs corrects / champs attendus, sur les traces où crm_attendu n'est pas None
    return 0.0

ATT = {"nom": "A B", "taille_entreprise": 8, "budget_mensuel": 25000, "delai": "1_mois"}
FIXTURE_CRM = [
    {"crm_attendu": ATT, "crm_saisi": dict(ATT)},
    {"crm_attendu": ATT, "crm_saisi": {**ATT, "taille_entreprise": 9, "delai": "3_mois"}},
    {"crm_attendu": None, "crm_saisi": None},
]
r = exactitude_crm(FIXTURE_CRM)
print(("ok       " if abs(r - 0.75) < 1e-9 else "À REVOIR  ") + f"exactitude_crm de la fixture : {r:.3f} (attendu 0.750)")
r2 = exactitude_crm([{"crm_attendu": ATT, "crm_saisi": None}])
print(("ok       " if r2 == 0 else "À REVOIR  ") + f"fiche jamais créée : {r2:.3f} (attendu 0.000)")

### 12.6 La latence : médiane et p95

La **moyenne** cache les mauvais cas. Une médiane de 1 s avec un p95 de 4 s veut dire qu'**un tour sur vingt** fait attendre plus de 4 secondes : le prospect s'en souvient. On rapporte donc :
- la **médiane** (p50) : le tour **typique** ;
- le **p95** : le seuil que **95 % des tours** ne dépassent pas, c'est-à-dire les **mauvais** cas.

On calcule sur **tous les tours de tous les appels** (pas sur la moyenne par appel).

**[Guidé] Exercice 12.3** : écrivez `percentile(valeurs, p)` **sans** `np.percentile`, par interpolation linéaire : on trie, on prend la position `(p / 100) * (n - 1)`, et on interpole entre les deux valeurs voisines. Puis `latences(traces)` qui renvoie `(médiane, p95)` sur tous les tours.

*Indice* : `i = int(pos)` ; `frac = pos - i` ; résultat = `v[i] + frac * (v[i + 1] - v[i])` (attention au dernier indice).

In [ ]:
def percentile(valeurs, p):
    # TODO : tri, position (p / 100) * (n - 1), interpolation linéaire
    return 0.0

def latences(traces):
    # TODO : rassembler tous les tours, renvoyer (percentile 50, percentile 95)
    return (0.0, 0.0)

jeu = [100, 200, 300, 400, 500]
tests = [(percentile(jeu, 50), 300), (percentile(jeu, 95), 480), (percentile(jeu, 0), 100), (percentile(jeu, 100), 500)]
for obtenu, attendu in tests:
    print(("ok       " if abs(obtenu - attendu) < 1e-9 else "À REVOIR  ") + f"obtenu {obtenu}, attendu {attendu}")
ref = np.percentile(jeu, 95)
print("Contrôle avec numpy :", ref)

### 12.7 Le WER sur les noms et les numéros

Comme au module 10, le WER global compte tous les mots pareil. Pour Awa, ce qui compte, c'est la **phrase d'identification** (« je m'appelle ... et mon numéro est le ... ») et surtout le **numéro** : un chiffre faux, et la fiche est inutilisable. On mesure donc deux choses sur cette phrase :
- le **WER** moyen (reprise du module 5 ci-dessous) ;
- le **taux de numéros exacts** : la part des appels où les 9 chiffres du numéro sont **tous** corrects.

In [ ]:
def normaliser(texte):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s'’-]", " ", texte.lower())).strip()

def wer(ref, hyp):
    r, h = normaliser(ref).split(), normaliser(hyp).split()
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prec, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            cout = 0 if r[i - 1] == h[j - 1] else 1
            prec, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prec + cout)
    return d[len(h)] / max(len(r), 1)

def compacter(s):
    return re.sub(r"[\s.\-]", "", s.lower())

print("WER de reprise :", round(wer("mon numéro est le 77 616", "mon numéro est le 77 617"), 2), "(attendu 0.17 : 1 mot sur 6)")

**[Guidé] Exercice 12.4** : écrivez `taux_numero_exact(traces)`, la part des traces où le numéro (`t["tel"]`) se retrouve **entièrement** dans l'hypothèse (`t["hyp_id"]`), en ignorant espaces, points et tirets (`compacter`). Puis `wer_identification(traces)` : le **WER moyen** de `ref_id` contre `hyp_id`.

*Indice* : `compacter(t["tel"]) in compacter(t["hyp_id"])` pour le numéro ; `np.mean([wer(...)])` pour le WER.

In [ ]:
def taux_numero_exact(traces):
    # TODO : part des traces où le numéro est entièrement retrouvé dans l'hypothèse
    return 0.0

def wer_identification(traces):
    # TODO : WER moyen de ref_id contre hyp_id
    return 0.0

FIXTURE_ID = [
    {"tel": "77 616 06 16", "ref_id": "mon numéro est le 77 616 06 16", "hyp_id": "mon numéro est le 77 616 06 16"},
    {"tel": "77 616 06 16", "ref_id": "mon numéro est le 77 616 06 16", "hyp_id": "mon numéro est le 77 616 06 17"},
]
a, b = taux_numero_exact(FIXTURE_ID), wer_identification(FIXTURE_ID)
print(("ok       " if abs(a - 0.5) < 1e-9 else "À REVOIR  ") + f"taux_numero_exact : {a:.2f} (attendu 0.50)")
print(("ok       " if abs(b - (0 + 1 / 8) / 2) < 1e-9 else "À REVOIR  ") + f"wer_identification : {b:.3f} (attendu 0.062)")

### 12.8 Le coût par appel

Le **coût par appel** dépend de la **durée** et des **tarifs** de chaque brique. Les tarifs ci-dessous sont **fictifs** (en F CFA par minute d'appel, repère indicatif) : ils servent à la **méthode**. Les vrais tarifs changent souvent ; vérifiez-les chez vos fournisseurs, et recalculez avec **votre** volume. La cascade additionne trois lignes (STT, LLM, TTS), le S2S une seule (audio).

**[Guidé] Exercice 12.5** : écrivez `cout_par_appel(trace, tarifs)` : durée en minutes (`trace["minutes"]`) multipliée par la **somme** des tarifs de l'architecture (`tarifs[trace["archi"]]`). Puis `cout_moyen(traces, tarifs)`.

In [ ]:
TARIFS = {   # FICTIFS, en F CFA par minute d'appel
    "cascade": {"stt": 8, "llm": 15, "tts": 22},
    "s2s": {"audio": 75},
}

def cout_par_appel(trace, tarifs):
    # TODO : minutes x somme des tarifs de l'architecture
    return 0.0

def cout_moyen(traces, tarifs):
    # TODO : moyenne de cout_par_appel
    return 0.0

t1 = {"archi": "cascade", "minutes": 2.0}
t2 = {"archi": "s2s", "minutes": 2.0}
for t, attendu in ((t1, 90.0), (t2, 150.0)):
    r = cout_par_appel(t, TARIFS)
    print(("ok       " if abs(r - attendu) < 1e-9 else "À REVOIR  ") + f"{t['archi']:8} 2 min -> {r} F CFA (attendu {attendu})")

### 12.9 Le tableau de bord

On rassemble les métriques dans une fonction `metriques(traces)`, et on affiche **les deux Awa côte à côte**. La **conformité** (aucune remise inventée) est une métrique de **plancher** : un seul manquement la fait passer sous 100 %.

In [ ]:
def metriques(traces):
    med, p95 = latences(traces)
    objections = [t for t in traces if t["type"] == "objection"]
    return {
        "rdv": taux_rdv(traces), "crm": exactitude_crm(traces), "numero": taux_numero_exact(traces), "wer": wer_identification(traces),
        "lat_med": med, "lat_p95": p95, "cout": cout_moyen(traces, TARIFS),
        "conformite": 1 - (sum(t["remise_inventee"] for t in objections) / len(objections) if objections else 0),
        "transfert": (sum(t["transfert_fait"] for t in traces if t["transfert_attendu"]) / max(1, sum(t["transfert_attendu"] for t in traces))),
    }

M = {a: metriques(t) for a, t in TRACES.items()}
LIGNES = [("rdv", "taux de rendez-vous", "{:.0%}"), ("crm", "exactitude CRM", "{:.0%}"), ("numero", "numéros exacts", "{:.0%}"),
          ("wer", "WER identification", "{:.1%}"), ("lat_med", "latence médiane (ms)", "{:.0f}"), ("lat_p95", "latence p95 (ms)", "{:.0f}"),
          ("cout", "coût par appel (F CFA)", "{:.0f}"), ("conformite", "conformité (pas de remise)", "{:.0%}"), ("transfert", "transferts réussis", "{:.0%}")]
print(f"{'métrique':30}" + "".join(f"{a:>12}" for a in M))
for cle, nom, fmt in LIGNES:
    print(f"{nom:30}" + "".join(f"{fmt.format(M[a][cle]):>12}" for a in M))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
couleurs = {"cascade": NAVY, "s2s": TEAL}
etiq = ["rdv", "crm", "numero", "conformite"]
for i, a in enumerate(M):
    axes[0].bar(np.arange(4) + (i - 0.5) * 0.35, [M[a][k] * 100 for k in etiq], 0.35, label=a, color=couleurs[a])
axes[0].set_xticks(range(4)); axes[0].set_xticklabels(["rdv", "CRM", "numéros", "conformité"]); axes[0].set_ylim(0, 105); axes[0].set_title("Succès (%)"); axes[0].legend(frameon=False)
for i, a in enumerate(M):
    axes[1].bar(np.arange(2) + (i - 0.5) * 0.35, [M[a]["lat_med"], M[a]["lat_p95"]], 0.35, color=couleurs[a])
axes[1].set_xticks(range(2)); axes[1].set_xticklabels(["médiane", "p95"]); axes[1].set_title("Latence par tour (ms)")
axes[2].bar(list(M), [M[a]["cout"] for a in M], color=[couleurs[a] for a in M]); axes[2].set_title("Coût par appel (F CFA, fictif)")
plt.tight_layout(); plt.show()

### 12.10 Combien de confiance accorder à ces chiffres ?

Avec **20 scénarios**, un écart de quelques points est **du bruit**. Un **intervalle bootstrap** le montre : on **rééchantillonne** les résultats avec remise des centaines de fois, et on regarde la dispersion de la moyenne. S'il est **large** et que les intervalles des deux agents se **chevauchent**, on ne peut pas dire qu'un agent est meilleur.

In [ ]:
def ic_bootstrap(valeurs, n=2000, graine=5):
    """Intervalle à 95 % de la moyenne par bootstrap."""
    rng = np.random.default_rng(graine)
    v = np.array(valeurs, dtype=float)
    moyennes = [rng.choice(v, size=len(v), replace=True).mean() for _ in range(n)]
    return float(np.percentile(moyennes, 2.5)), float(np.percentile(moyennes, 97.5))

for a, traces in TRACES.items():
    res = [t["rdv_reserve"] for t in traces if t["rdv_attendu"]]
    bas, haut = ic_bootstrap(res)
    print(f"{a:8} taux de rendez-vous {np.mean(res):.0%}, intervalle à 95 % : de {bas:.0%} à {haut:.0%}  (n = {len(res)})")

**Lecture** : les intervalles sont **larges** (30 à 45 points de large) et se **chevauchent**. Avec 16 rendez-vous attendus, on ne peut **pas** conclure sur le taux de rendez-vous. Pour trancher, il faut **plus de scénarios** (des dizaines par type), et **plusieurs essais** par scénario, car un LLM ne répond pas toujours pareil. La **latence**, qui compte des dizaines de tours, est bien plus stable que les taux de succès.

### 12.11 [Avancé] Un LLM juge

Certaines qualités ne se mesurent pas par une règle : **ton poli**, **réponse claire**, **honnêteté**. Un **LLM juge** lit la transcription et note selon une **rubrique**. Précautions :
- une rubrique **précise**, avec une **échelle** (1 à 5) et des **exemples** ;
- une sortie **structurée** (JSON), **vérifiée** : une réponse mal formée n'est **pas** un 3 ;
- mesurer l'**accord avec un humain** sur un échantillon, **avant** de lui faire confiance ;
- le juge a ses **biais** (il préfère souvent les réponses longues) ; on ne l'utilise **pas** seul pour une exigence de **plancher** comme la conformité (là, une **règle** vaut mieux).

**[Avancé] Exercice 12.6** : écrivez
1. `RUBRIQUE`, un **texte** de consigne pour le juge (il doit mentionner les trois critères `politesse`, `exactitude`, `conformite`, l'échelle de 1 à 5 et le format JSON attendu) ;
2. `parser_note(texte)`, qui lit la **réponse du juge** et renvoie un dictionnaire `{"politesse": int, "exactitude": int, "conformite": int}`, ou `None` si la réponse est inutilisable. Elle doit accepter du JSON **seul**, du JSON entouré de **texte** ou de balises de code, et refuser une note **hors de 1 à 5** ou un **critère manquant** ;
3. `accord(notes_juge, notes_humain, tolerance=1)`, la part des critères où le juge est à **tolérance près** de l'humain (listes de dictionnaires de même longueur ; les juges `None` comptent comme désaccord).

*Indice* : pour extraire le JSON, cherchez la première `{` et la dernière `}` avec `str.find` et `str.rfind`, puis `json.loads` dans un `try`.

In [ ]:
RUBRIQUE = ""   # TODO : consigne du juge (critères, échelle 1 à 5, format JSON)

def parser_note(texte):
    # TODO : extraire le JSON, vérifier les trois critères et l'échelle ; None sinon
    return None

def accord(notes_juge, notes_humain, tolerance=1):
    # TODO : part des critères où |juge - humain| <= tolerance ; juge None = désaccord
    return 0.0

def tester_juge():
    echecs = []
    if not all(m in RUBRIQUE.lower() for m in ("politesse", "exactitude", "conformite", "json", "1", "5")):
        echecs.append("RUBRIQUE : citer les trois critères, l'échelle 1 à 5 et JSON")
    ok = {"politesse": 5, "exactitude": 4, "conformite": 5}
    if parser_note('{"politesse": 5, "exactitude": 4, "conformite": 5}') != ok: echecs.append("JSON seul")
    if parser_note('Voici mon évaluation : {"politesse": 5, "exactitude": 4, "conformite": 5} Merci.') != ok: echecs.append("JSON entouré de texte")
    if parser_note('```json\n{"politesse": 5, "exactitude": 4, "conformite": 5}\n```') != ok: echecs.append("balises de code")
    if parser_note('{"politesse": 7, "exactitude": 4, "conformite": 5}') is not None: echecs.append("note hors de 1 à 5")
    if parser_note('{"politesse": 5, "exactitude": 4}') is not None: echecs.append("critère manquant")
    if parser_note("je ne sais pas") is not None: echecs.append("pas de JSON")
    h = [{"politesse": 5, "exactitude": 4, "conformite": 5}, {"politesse": 3, "exactitude": 3, "conformite": 5}]
    j = [{"politesse": 4, "exactitude": 2, "conformite": 5}, None]
    if abs(accord(j, h) - 2 / 6) > 1e-9: echecs.append("accord : 2 critères sur 6 (tolérance 1)")
    print("Tous les tests passent." if not echecs else "À REVOIR : " + " ; ".join(echecs))

tester_juge()

**Un juge simulé.** Pour tester le circuit **sans clé**, un faux juge note d'après des **mots-clés** de la transcription et renvoie du texte **comme un vrai LLM** (parfois bien formé, parfois non). Il reste **fictif** : il sert à vérifier votre `parser_note` et à calculer un accord, pas à juger Awa.

In [ ]:
TRANSCRIPTIONS = [
    ("Awa : Je comprends. Le Pro est à 49 000 F CFA par mois, avec 2 mois offerts en paiement annuel.", {"politesse": 5, "exactitude": 5, "conformite": 5}),
    ("Awa : Je peux vous faire 20 % de remise si vous signez aujourd'hui.", {"politesse": 4, "exactitude": 2, "conformite": 1}),
    ("Awa : Ça coûte 30 000 francs, c'est tout. Décidez vite.", {"politesse": 2, "exactitude": 1, "conformite": 2}),
    ("Awa : Bien sûr, je vous envoie un e-mail. Souhaitez-vous aussi un rendez-vous de 30 minutes ?", {"politesse": 5, "exactitude": 5, "conformite": 5}),
]

def juge_simule(transcription, i):
    """Faux LLM : note d'après des mots-clés, avec un format de réponse qui varie."""
    t = transcription.lower()
    conf = 1 if ("%" in t or "remise" in t) else (2 if "30 000" in t else 5)
    pol = 2 if "décidez vite" in t else 5
    exa = 5 if ("49 000" in t or "e-mail" in t) else 2
    notes = json.dumps({"politesse": pol, "exactitude": exa, "conformite": conf})
    return [f"Voici mon évaluation : {notes}", f"```json\n{notes}\n```", notes, "Je ne peux pas évaluer cet appel."][i % 4]

notes_juge = [parser_note(juge_simule(t, i)) for i, (t, _) in enumerate(TRANSCRIPTIONS)]
notes_humain = [h for _, h in TRANSCRIPTIONS]
print("Notes du juge :", notes_juge)
print(f"Accord avec l'humain (tolérance 1) : {accord(notes_juge, notes_humain):.0%}")

In [ ]:
def juge_reel(transcription, modele="gpt-4.1-mini"):
    """Optionnel : appelle un vrai LLM juge (clé OpenAI requise). Non testé avec une vraie clé au moment de l'écriture."""
    if not CLE_OPENAI:
        print("Pas de clé OpenAI : étape optionnelle ignorée.")
        return None
    from openai import OpenAI
    reponse = OpenAI().chat.completions.create(model=modele, temperature=0, response_format={"type": "json_object"},
                                               messages=[{"role": "system", "content": RUBRIQUE}, {"role": "user", "content": transcription}])
    return parser_note(reponse.choices[0].message.content)

print(juge_reel(TRANSCRIPTIONS[0][0]))

Observez : le juge simulé a produit une réponse **inutilisable** (« Je ne peux pas évaluer cet appel. ») et votre `parser_note` l'a bien refusée plutôt que de lui inventer une note. Dans une vraie évaluation, on **relance** le juge (une ou deux fois) et on **compte** les échecs. Et la **conformité** reste mesurée par une **règle** (module 10), pas par le juge.

### 12.12 Choisir à partir des chiffres

On ne choisit pas une architecture parce qu'elle a **plus** de points dans un tableau : on l'évalue contre des **exigences écrites à l'avance**. Pour Awa, par exemple :

| Exigence | Seuil |
|---|---|
| Taux de rendez-vous | au moins 80 % |
| Exactitude CRM | au moins 95 % |
| Latence p95 | au plus 3 000 ms |
| Conformité (remise) | 100 % (plancher) |
| Coût par appel | au plus 200 F CFA (fictif) |

Une architecture qui **échoue** à une exigence de **plancher** (conformité) est **écartée**, même si elle est moins chère. Parmi celles qui **réussissent tout**, on prend la **moins chère**.

**[Guidé] Exercice 12.7** : écrivez `verifier(metriques_archi, exigences)` qui renvoie un dictionnaire `{critère: bool}`, puis `decider(toutes_metriques, exigences)` qui renvoie le **nom** de l'architecture retenue (celle qui réussit **tous** les critères, la moins chère en cas de pluralité) ou `"aucune"`.

*Indice* : pour les critères en `min`, on teste `>=` ; pour ceux en `max`, `<=`. Les clés des exigences sont `rdv_min`, `crm_min`, `p95_max`, `conformite_min`, `cout_max`.

In [ ]:
EXIGENCES = {"rdv_min": 0.80, "crm_min": 0.95, "p95_max": 3000, "conformite_min": 1.0, "cout_max": 200}

def verifier(m, exigences):
    # TODO : {"rdv": bool, "crm": bool, "p95": bool, "conformite": bool, "cout": bool}
    return {}

def decider(toutes, exigences):
    # TODO : parmi les architectures qui réussissent tout, la moins chère ; sinon "aucune"
    return "aucune"

FIX = {
    "a": {"rdv": 0.9, "crm": 0.97, "lat_p95": 2800, "conformite": 1.0, "cout": 150},
    "b": {"rdv": 0.9, "crm": 0.97, "lat_p95": 2000, "conformite": 1.0, "cout": 120},
    "c": {"rdv": 0.95, "crm": 0.99, "lat_p95": 1500, "conformite": 0.8, "cout": 50},
}
v = verifier(FIX["c"], EXIGENCES)
print(("ok       " if v.get("conformite") is False and v.get("rdv") is True else "À REVOIR  ") + f"verifier(c) : {v}")
d = decider(FIX, EXIGENCES)
print(("ok       " if d == "b" else "À REVOIR  ") + f"decider : {d} (attendu b : la moins chère qui réussit tout, c est écartée par la conformité)")
print(("ok       " if decider({"c": FIX["c"]}, EXIGENCES) == "aucune" else "À REVOIR  ") + "aucune architecture ne réussit : « aucune »")

### 12.13 Atelier : choisir entre cascade et speech-to-speech

Appliquez `verifier` et `decider` à **vos** deux Awa simulées (`M`), puis à **vos** exigences.

In [ ]:
for a, m in M.items():
    print(f"{a:8}", {k: ("réussi" if ok else "ÉCHEC") for k, ok in verifier(m, EXIGENCES).items()})
print("\nArchitecture retenue avec les exigences par défaut :", decider(M, EXIGENCES), "(« aucune » est une réponse valable : elle dit qu'il faut améliorer avant de choisir)")

**Questions pour l'atelier** (à répondre dans une cellule de texte) :
1. **Quelles exigences** gardez-vous, modifiez-vous ou ajoutez-vous pour **Baobab**, et **pourquoi** ? Écrivez-les **avant** de regarder le tableau.
2. Quelle architecture est retenue, et **sur quels critères** ? **Si vous changez un seuil de 5 points**, la décision change-t-elle ?
3. Au vu des **intervalles** du 12.10, **que pouvez-vous affirmer** et **que ne pouvez-vous pas affirmer** ?
4. **Quel scénario ajoutez-vous** au jeu pour départager les architectures, et **combien d'essais** par scénario prévoyez-vous ?
5. Quelle **métrique** est un **plancher** (un seul manquement suffit à écarter), et pourquoi ne la confiez-vous **pas** au LLM juge ?

**Livrable** : le choix argumenté (une demi-page) avec le tableau des métriques et les exigences. Il alimentera la **grille de notation** du projet final.

### À retenir

- Évaluer, c'est **définir le succès avant de mesurer**, écrire un **jeu de scénarios**, enregistrer des **traces**, calculer des **métriques**, décider avec des **exigences écrites à l'avance**.
- Les métriques d'Awa : **taux de rendez-vous**, **exactitude CRM** (champ par champ), **latence médiane et p95**, **WER sur les noms et numéros**, **coût par appel**, plus une **conformité** de plancher.
- La **moyenne** cache les mauvais cas : on rapporte la **médiane** et le **p95**, sur **tous les tours**.
- Avec **20 scénarios**, les écarts de quelques points sont du **bruit** : on calcule un **intervalle** (bootstrap) et on augmente le jeu et le nombre d'essais.
- Un **LLM juge** a besoin d'une **rubrique**, d'une sortie **vérifiée** et d'un **accord mesuré** avec un humain. Les **planchers** se contrôlent par **règle**.
- On **choisit** l'architecture avec des **exigences** : celle qui réussit **tout** et coûte **le moins**, sinon on **améliore** avant de choisir.

---
### Quiz du Module 12

**Q1.** Pourquoi rapporter le p95 de la latence en plus de la médiane ?
a) parce qu'il est toujours plus petit que la médiane · b) parce que la médiane est impossible à calculer · c) parce que la moyenne suffit · d) parce qu'il montre les mauvais cas : 5 % des tours dépassent cette valeur

**Q2.** Un appel se termine sans que la fiche CRM soit créée alors qu'elle était attendue. Comment compter l'exactitude ?
a) ne pas compter cet appel · b) tous les champs attendus de cet appel comptent comme faux · c) compter les champs comme corrects par défaut · d) compter la moitié des champs

**Q3.** Pourquoi mesure-t-on le taux de numéros exacts en plus du WER global ?
a) parce qu'un seul chiffre faux rend la fiche inutilisable, alors que le WER traite tous les mots pareil · b) parce que le WER ne se calcule pas sur les numéros · c) parce que le WER est toujours trop optimiste · d) parce que les numéros ne contiennent jamais d'erreurs

**Q4.** Les intervalles de confiance de deux agents sur 16 rendez-vous attendus se chevauchent largement. Que conclure ?
a) l'agent avec le plus grand taux est meilleur · b) les deux agents sont identiques · c) on ne peut pas conclure : il faut plus de scénarios et plusieurs essais par scénario · d) il faut supprimer l'agent le moins cher

**Q5.** Pourquoi ne pas confier la métrique « aucune remise inventée » à un LLM juge ?
a) parce qu'un LLM ne sait pas lire le français · b) parce qu'elle ne se mesure pas · c) parce que c'est trop long · d) parce que c'est un plancher : un seul manquement compte, et une règle explicite est plus fiable qu'un juge qui a ses biais

In [ ]:
mes_reponses_m12 = {1: "", 2: "", 3: "", 4: "", 5: ""}   # TODO : vos réponses

In [ ]:
def corriger_quiz_module12(reponses):
    attendu = {1: "d", 2: "b", 3: "a", 4: "c", 5: "d"}
    explications = {
        1: "La médiane décrit le tour typique ; le p95 décrit les mauvais cas, que le prospect remarque et retient.",
        2: "Pas de fiche : tous les champs attendus sont faux. Ne pas compter l'appel cacherait l'échec le plus grave.",
        3: "Un chiffre faux dans le numéro rend la fiche inutilisable ; le WER donne le même poids à tous les mots.",
        4: "Avec 16 cas, l'incertitude est large : on ne peut pas départager. Il faut plus de scénarios et plusieurs essais.",
        5: "La conformité est un plancher : on la contrôle par une règle explicite, et le juge sert aux qualités plus floues, après mesure de son accord avec un humain.",
    }
    score_quiz = 0
    for q in attendu:
        r = str(reponses.get(q, "")).strip().lower()
        ok = r == attendu[q]; score_quiz += ok
        print(f"Q{q} : {'correct' if ok else 'à revoir'} (votre réponse : {r or '-'}). {explications[q]}")
    print(f"\nScore : {score_quiz}/5")

corriger_quiz_module12(mes_reponses_m12)

---
## 13. Production, sécurité et conformité : mettre Awa en service

**Objectifs**
1. Choisir une **option de déploiement** (LiveKit Cloud, Pipecat Cloud, auto-hébergement) et connaître les commandes.
2. Comprendre la **téléphonie SIP** et le déroulé d'un **appel sortant**, avec ses échecs.
3. **Estimer le coût mensuel** d'Awa et voir comment il varie avec le volume.
4. Exécuter une **checklist de sécurité et de conformité** (clés, annonce de l'IA, horaires, conservation).
5. **[Avancé]** **Masquer les données personnelles** dans les journaux.
6. Préparer l'**atelier** : déployer Awa sur un numéro de test et passer un appel sortant.

**Durée** : 30 minutes d'exercices, puis l'atelier (20 minutes) et le quiz (10 minutes). Tout ce qui est exécuté ici est **hors ligne** : le notebook **ne déploie rien** et **n'appelle personne**. Les commandes de déploiement et d'appel sont **montrées**, pour l'atelier.

**Avertissement sur le volet légal** : les repères sur l'**annonce de l'IA**, le **consentement**, les **horaires d'appel** et la **durée de conservation** sont des **repères de bonne pratique**, **pas un avis juridique**. Je n'ai **pas pu relire le texte de la loi sénégalaise** (la loi n° 2008-12 du 25 janvier 2008 sur la protection des données à caractère personnel est le point de départ, avec la Commission de protection des données personnelles, la CDP). **Faites relire ce volet par le juridique avant toute mise en production.**

In [ ]:
# Cellule de reprise du Module 13
import re, json, math, logging, io
import datetime as dt
import numpy as np, matplotlib.pyplot as plt
try:
    NAVY
except NameError:
    raise RuntimeError("Exécutez d'abord la section 0 (Configuration).")
RED = "#C0392B"
print("Module 13 prêt.")

### 13.1 Où faire tourner Awa ?

Jusqu'ici Awa tournait sur **votre machine** (`console`, `dev`). En production, elle tourne sur un **serveur** qui reste allumé et qui reçoit les appels. Trois options :

| Option | Principe | Commandes et fichiers (documentation) | À retenir |
|---|---|---|---|
| **LiveKit Cloud** | LiveKit héberge l'agent | `lk agent create`, `lk agent deploy` ; `Dockerfile` et `livekit.toml` ; secrets avec `--secrets` ou `--secrets-file` | même framework qu'Awa : **aucune réécriture** |
| **Pipecat Cloud** | Pipecat héberge l'agent | `pipecat cloud deploy` ; `pcc-deploy.toml` avec `secret_set` ; profils `agent-1x`, `agent-2x`, `agent-3x` | Awa est écrite avec **LiveKit Agents** : passer à Pipecat veut dire **réécrire** l'agent |
| **Auto-hébergement** | vous faites tourner serveur et agent | serveur LiveKit open source, agent dans un conteneur | **contrôle total** (données, région), mais **vous exploitez** : mises à jour, supervision, astreinte |

Autres commandes LiveKit Cloud utiles : `lk agent deploy --deployment staging` pour déployer d'abord en **préproduction**, `lk agent promote --deployment staging` pour la passer en **production** sans reconstruire, `lk agent versions` et `lk agent rollback` pour **revenir en arrière**. `lk agent dockerfile` génère un `Dockerfile` adapté : **utilisez-le** plutôt que d'en copier un.

*Sources : documentation LiveKit (CLI `lk agent`, déploiement) et documentation Pipecat Cloud (déploiement), consultées pour ce module ; la liste des régions et les tarifs ne sont pas repris ici : **vérifiez-les** à la source. L'API évolue vite.*

**Critères de choix** pour Baobab : où sont les **appelants** (latence réseau), où doivent rester les **données vocales** (conformité), qui **exploite** (équipe, astreinte), et le **coût** (13.3).

In [ ]:
# Commandes de l'atelier (à lancer sur VOTRE machine, pas ici). Elles sont affichées, pas exécutées.
COMMANDES_LIVEKIT = """
# 1) installer et connecter la CLI LiveKit, puis choisir le projet (voir la documentation)
# 2) à la racine du projet Awa (awa_agent.py, baobab_api.py, tts_normalisation.py, requirements.txt)
lk agent create --region <region> --secrets-file .env.production
lk agent deploy --deployment staging
lk agent promote --deployment staging      # quand la préproduction est validée
lk agent versions --id <AGENT_ID>
lk agent rollback --id <AGENT_ID> --version <VERSION>   # en cas de problème
"""
print(COMMANDES_LIVEKIT)
print("Règle d'or : .env.production n'est JAMAIS ajouté au dépôt (voir 13.4).")

### 13.2 La téléphonie SIP et l'appel sortant

Pour que le **réseau téléphonique** parle à LiveKit, on passe par le protocole **SIP** et un **opérateur** (un fournisseur de trunk SIP) :
- un **numéro entrant** : les prospects appellent Awa ;
- un **trunk sortant** : Awa appelle un prospect. Le trunk a un **identifiant** (`sip_trunk_id`, visible avec `lk sip outbound list`).

**Déroulé d'un appel sortant avec LiveKit** (documentation LiveKit, appels sortants) :
1. On **répartit** l'agent avec le numéro dans les **métadonnées** : `lk dispatch create --new-room --agent-name <nom> --metadata '{"phone_number": "+221..."}'`. L'agent doit avoir un **nom** (`agent_name`).
2. L'agent lit le numéro dans `ctx.job.metadata`, puis appelle `ctx.api.sip.create_sip_participant(api.CreateSIPParticipantRequest(room_name=ctx.room.name, sip_trunk_id=..., sip_call_to=numero, participant_identity=numero, wait_until_answered=True))`.
3. Si l'appel échoue, une exception `api.SipCallError` porte `sip_status_code` : **486 ou 603** = le correspondant a **refusé** ; **408 ou 480** = **pas de réponse** ou indisponible ; **5xx** = **panne** côté trunk ou opérateur. Dans les deux derniers cas, il faut **libérer la tâche** avec `ctx.shutdown()`.
4. Si le correspondant décroche : `ctx.wait_for_participant(identity=numero)`, puis on démarre la session.

**Deux pièges** :
- Une **messagerie vocale** répond comme un humain (`200 OK`) : `wait_until_answered` **ne détecte pas** la messagerie. Il faut une **détection de répondeur** à part.
- Sur un appel sortant, **le correspondant parle en premier** (c'est lui qui décroche), puis Awa **s'annonce**.

**[Guidé] Exercice 13.1** : écrivez `classer_echec_sip(code)` qui renvoie `"refuse"` (486, 603), `"indisponible"` (408, 480), `"panne_operateur"` (500 à 599) ou `"autre"`. Puis `numero_valide(numero)` : un mobile sénégalais au format international, `+221` suivi de **9 chiffres** commençant par **70, 75, 76, 77 ou 78** (espaces, points et tirets ignorés).

*Indice* : pour le numéro, nettoyez avec `re.sub(r"[\s.-]", "", numero)` puis `re.fullmatch(...)`.

In [ ]:
def classer_echec_sip(code):
    # TODO : "refuse", "indisponible", "panne_operateur" ou "autre"
    return "autre"

def numero_valide(numero):
    # TODO : +221 puis 70, 75, 76, 77 ou 78, puis 7 chiffres
    return False

tests = [(classer_echec_sip(486), "refuse"), (classer_echec_sip(603), "refuse"), (classer_echec_sip(480), "indisponible"),
         (classer_echec_sip(408), "indisponible"), (classer_echec_sip(503), "panne_operateur"), (classer_echec_sip(404), "autre"),
         (numero_valide("+221776160616"), True), (numero_valide("+221 77 616 06 16"), True), (numero_valide("+221 78-245-31-09"), True),
         (numero_valide("776160616"), False), (numero_valide("+33612345678"), False), (numero_valide("+221331234567"), False)]
echecs = [i + 1 for i, (o, a) in enumerate(tests) if o != a]
print("Tous les tests passent." if not echecs else f"À REVOIR : tests {echecs}")

### 13.3 Le coût mensuel

Au module 12, nous avons calculé un **coût par appel**. En production, on veut un **coût par mois** : le **volume** d'appels, la **durée**, la **téléphonie** (le trunk facture aussi la minute), et un **coût fixe** (hébergement, supervision).

**Coût mensuel** = appels par jour × jours ouvrés × minutes moyennes × (tarif de l'architecture + tarif de téléphonie) + coût fixe.

Les tarifs ci-dessous sont **fictifs** (**repère indicatif**) : vérifiez ceux de **vos** fournisseurs, y compris le prix de la minute **sortante vers un mobile sénégalais**, qui varie selon l'opérateur. Les appels **non décrochés** ou tombés sur **messagerie** peuvent aussi être facturés : ce modèle simple les **ignore**.

**[Guidé] Exercice 13.2** : écrivez `cout_mensuel(appels_par_jour, minutes, jours, archi, tarifs, fixe)`.

*Indice* : `tarifs[archi]` et `tarifs["telephonie"]` sont des F CFA par minute.

In [ ]:
TARIFS_M13 = {"cascade": 45, "s2s": 75, "telephonie": 15}   # FICTIFS, F CFA par minute

def cout_mensuel(appels_par_jour, minutes, jours, archi, tarifs, fixe):
    # TODO : volume en minutes x (tarif archi + téléphonie) + fixe
    return 0.0

r = cout_mensuel(10, 2.0, 20, "cascade", TARIFS_M13, 10000)
print(("ok       " if abs(r - 34000) < 1e-6 else "À REVOIR  ") + f"fixture : {r:.0f} F CFA (attendu 34000)")

In [ ]:
# Scénario de Baobab : 20 appels par jour, 3 minutes, 22 jours, coût fixe fictif de 25 000 F CFA
for a in ("cascade", "s2s"):
    print(f"{a:8} : {cout_mensuel(20, 3.0, 22, a, TARIFS_M13, 25000):>9,.0f} F CFA par mois".replace(",", " "))

volumes = np.arange(0, 81, 5)
fig, ax = plt.subplots(figsize=(7, 3.4))
for a, c in (("cascade", NAVY), ("s2s", TEAL)):
    ax.plot(volumes, [cout_mensuel(v, 3.0, 22, a, TARIFS_M13, 25000) / 1000 for v in volumes], color=c, lw=2.5, label=a)
ax.set_xlabel("appels par jour"); ax.set_ylabel("milliers de F CFA par mois (fictif)"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

**Lecture** : le coût **monte linéairement** avec le volume ; l'écart entre les deux architectures **grandit** avec lui, car il vient du tarif par minute. À faible volume, le **coût fixe** domine. Avant de choisir, **calculez avec votre volume réel** : c'est le résultat du module 12 (exigence de coût) qui devient un **budget mensuel**.

### 13.4 Sécurité : les clés et les données vocales

Les **clés** (LiveKit, OpenAI, trunk SIP) valent de l'argent : une clé publiée par erreur est utilisée **par d'autres** en quelques minutes. Les **données vocales** et les **numéros** sont des **données personnelles**. Règles de base :

1. **Aucune clé dans le code** ni dans le dépôt : variables d'environnement ou **gestionnaire de secrets**.
2. Le fichier **`.env`** est dans **`.gitignore`**.
3. **Une clé par environnement** (test, production), **révocable** ; on la **fait tourner** régulièrement et **tout de suite** en cas de fuite.
4. **Moindre privilège** : une clé qui ne sert qu'à ce dont elle a besoin.
5. **Pas de données personnelles dans les journaux** (13.6), et une **durée de conservation** fixée pour les enregistrements et transcriptions.

Une **checklist exécutable** vérifie ces règles **automatiquement**, par exemple **avant chaque déploiement**. Voici un exemple de projet à auditer, en deux versions : un **mauvais** et un **bon**.

In [ ]:
MAUVAIS = {
    "fichiers": {"awa_agent.py": 'OPENAI_KEY = "sk-abc123def456ghi789"\nprint("appel de", numero)\n'},
    "gitignore": "__pycache__/\n",
    "env": {"LIVEKIT_URL": "wss://x.livekit.cloud"},
    "ouverture": "Bonjour, Awa de Baobab Software, avez-vous deux minutes ?",
    "horaires": {"jours": (0, 1, 2, 3, 4, 5, 6), "debut": 6, "fin": 22},
    "retention_jours": None, "filtre_pii": False, "liste_refus": None,
}
BON = {
    "fichiers": {"awa_agent.py": 'import os\nOPENAI_KEY = os.environ["OPENAI_API_KEY"]\n'},
    "gitignore": "__pycache__/\n.env\n.env.production\n",
    "env": {"LIVEKIT_URL": "wss://x.livekit.cloud", "LIVEKIT_API_KEY": "k", "LIVEKIT_API_SECRET": "s", "AWA_SIP_TRUNK_ID": "ST_x"},
    "ouverture": "Bonjour, ici Awa, l'assistante virtuelle de Baobab Software. Je suis une intelligence artificielle. Avez-vous deux minutes ?",
    "horaires": {"jours": (0, 1, 2, 3, 4), "debut": 9, "fin": 17},
    "retention_jours": 30, "filtre_pii": True, "liste_refus": set(),
}

MOTIF_CLE = re.compile(r"(sk-[A-Za-z0-9]{10,}|(?:api_key|api_secret|secret|password)\s*=\s*['\"][^'\"]{6,}['\"])", re.I)
VARIABLES_REQUISES = ("LIVEKIT_URL", "LIVEKIT_API_KEY", "LIVEKIT_API_SECRET", "AWA_SIP_TRUNK_ID")

def auditer_cles(projet):
    """Exemple fourni : contrôles C1 à C3 (clés et secrets)."""
    en_clair = [nom for nom, contenu in projet["fichiers"].items() if MOTIF_CLE.search(contenu)]
    lignes = [l.strip() for l in projet["gitignore"].splitlines()]
    manquantes = [v for v in VARIABLES_REQUISES if v not in projet["env"]]
    return [("C1", not en_clair, "aucune clé en clair dans le code" + (f" (trouvée dans {en_clair})" if en_clair else "")),
            ("C2", ".env" in lignes, ".env est dans .gitignore"),
            ("C3", not manquantes, "variables requises présentes" + (f" (manque {manquantes})" if manquantes else ""))]

for nom, p in (("MAUVAIS", MAUVAIS), ("BON", BON)):
    print(nom, [(i, "ok" if ok else "ECHEC") for i, ok, _ in auditer_cles(p)])

**[Guidé] Exercice 13.3** : écrivez `auditer_conformite(projet)` qui renvoie la liste `[(identifiant, ok, message), ...]` pour cinq contrôles :
- **C4** l'**ouverture** annonce une **IA** : elle contient « intelligence artificielle » ou « assistante virtuelle » (insensible à la casse) ;
- **C5** les **horaires** sont **raisonnables** : pas de dimanche (`6` absent des jours), début **à partir de 8 h**, fin **au plus tard à 18 h** (**repère indicatif**, à fixer avec le juridique) ;
- **C6** une **durée de conservation** est **fixée** (`retention_jours` non vide) et d'**au plus 90 jours** (**repère indicatif**) ;
- **C7** le **filtre** de données personnelles est **actif** (`filtre_pii` vrai) ;
- **C8** une **liste de refus** existe (`liste_refus` n'est pas `None`).

*Indice* : renvoyez une liste de cinq tuples ; pour la liste de refus, testez avec `is not None` (un ensemble vide est valable).

In [ ]:
def auditer_conformite(projet):
    # TODO : cinq tuples (identifiant, ok, message) pour C4 à C8
    return []

def rapport(projet):
    tout = auditer_cles(projet) + auditer_conformite(projet)
    for i, ok, msg in tout:
        print(f"  {i} {'ok    ' if ok else 'ECHEC '} {msg}")
    return all(ok for _, ok, _ in tout)

res_m = {i: ok for i, ok, _ in auditer_conformite(MAUVAIS)}
res_b = {i: ok for i, ok, _ in auditer_conformite(BON)}
attendu_m = {"C4": False, "C5": False, "C6": False, "C7": False, "C8": False}
attendu_b = {k: True for k in attendu_m}
print(("ok       " if res_m == attendu_m else "À REVOIR  ") + "projet MAUVAIS : cinq échecs attendus")
print(("ok       " if res_b == attendu_b else "À REVOIR  ") + "projet BON : cinq réussites attendues")

In [ ]:
print("Audit du projet MAUVAIS :"); feu_vert = rapport(MAUVAIS); print("  -> déploiement autorisé :", feu_vert)
print("Audit du projet BON :");     feu_vert = rapport(BON);     print("  -> déploiement autorisé :", feu_vert)

### 13.5 Annonce de l'IA, consentement et horaires

Trois pratiques pour un agent vocal **qui appelle des personnes** :

- **Annoncer l'IA** dès la première phrase, et ne **jamais nier** être une IA si on le demande. C'est dans le **prompt** d'Awa depuis ce module, et dans sa **phrase d'ouverture** (`PHRASE_OUVERTURE`).
- **Respecter le consentement** : prévenir si l'appel est **enregistré** ou **transcrit**, accepter un **refus** (« ne me rappelez plus ») et le **mémoriser** dans une **liste de refus** consultée **avant** chaque appel.
- **Appeler à des heures raisonnables** : jours ouvrés, journée. Les horaires d'Awa sont **lundi à vendredi, 9 h à 17 h** (**repère indicatif**), l'heure de Dakar étant UTC+0 toute l'année.

**À faire relire par le juridique** : le **fondement légal** de la prospection téléphonique, le **consentement** exigé, la **déclaration ou l'autorisation auprès de la CDP**, la **durée de conservation** des enregistrements, et le **transfert de données** vers des serveurs hors du Sénégal (c'est le cas des services cloud).

**[Guidé] Exercice 13.4** : écrivez `appel_autorise(maintenant, numero, liste_refus, horaires)` qui renvoie `(ok, raison)` après les contrôles, **dans cet ordre** : `numero_valide` (« numéro invalide »), liste de refus (« numéro sur la liste de refus »), **jour** (« jour non autorisé ») puis **heure** (« hors des horaires »). `maintenant` est un `datetime` à l'heure de Dakar ; `horaires` a les clés `jours`, `debut`, `fin`.

*Indice* : `maintenant.weekday()` vaut 0 pour lundi ; `maintenant.hour` est l'heure ; le début est **inclus**, la fin **exclue**. Les numéros de la liste de refus sont écrits **sans espaces**.

In [ ]:
HORAIRES = {"jours": (0, 1, 2, 3, 4), "debut": 9, "fin": 17}

def appel_autorise(maintenant, numero, liste_refus, horaires):
    # TODO : (ok, raison) dans l'ordre : numéro, liste de refus, jour, heure
    return True, "ok"

LUNDI_10H, DIMANCHE_10H, LUNDI_20H, LUNDI_17H = (dt.datetime(2026, 11, 2, 10), dt.datetime(2026, 11, 1, 10), dt.datetime(2026, 11, 2, 20), dt.datetime(2026, 11, 2, 17))
cas = [((LUNDI_10H, "+221776160616", set()), (True, "ok")),
       ((LUNDI_10H, "+33612345678", set()), (False, "numéro invalide")),
       ((LUNDI_10H, "+221776160616", {"+221776160616"}), (False, "numéro sur la liste de refus")),
       ((DIMANCHE_10H, "+221776160616", set()), (False, "jour non autorisé")),
       ((LUNDI_20H, "+221776160616", set()), (False, "hors des horaires")),
       ((LUNDI_17H, "+221776160616", set()), (False, "hors des horaires"))]
echecs = [i + 1 for i, ((m, n, l), att) in enumerate(cas) if appel_autorise(m, n, l, HORAIRES) != att]
print("Tous les tests passent." if not echecs else f"À REVOIR : cas {echecs}")

### 13.6 [Avancé] Masquer les données personnelles dans les journaux

Les **journaux** sont **copiés** (outils de supervision, e-mails d'alerte, captures d'écran) et **lus** par plus de monde que le CRM. Un numéro de téléphone ou un e-mail **en clair** dans un journal est une **fuite** possible. On **masque** à l'**écriture**, avant que la ligne n'existe.

**[Avancé] Exercice 13.5** : écrivez
1. `masquer_pii(texte)` : remplace les **e-mails** par `[EMAIL]` et les **numéros sénégalais** (avec ou sans `+221`, avec espaces, points, tirets ou collés) par `[TEL]`. Il **ne doit pas toucher** aux prix (« 49 000 F CFA »), aux dates (« 2026-11-03 ») ni aux durées (« 30 minutes ») ;
2. `FiltrePII`, un `logging.Filter` qui masque le message **avant** son écriture.

*Indice* : un numéro mobile commence par `70`, `75`, `76`, `77` ou `78` suivi de **7 chiffres** (en groupes de 2 ou 3). Utilisez des **marqueurs de bord** `(?<!\d)` et `(?!\d)` pour ne pas couper un nombre plus long. Pour le filtre : `record.msg = masquer_pii(record.getMessage())` et `record.args = ()`.

In [ ]:
def masquer_pii(texte):
    # TODO : e-mails -> [EMAIL], numéros sénégalais -> [TEL]
    return texte

class FiltrePII(logging.Filter):
    def filter(self, record):
        # TODO : masquer le message avant l'écriture
        return True

cas = [("Mon mail est a.b@x.sn", "Mon mail est [EMAIL]"),
       ("Rappelez le 77 616 06 16 demain", "Rappelez le [TEL] demain"),
       ("Numéro +221 78 245 31 09 ok", "Numéro [TEL] ok"),
       ("Collé : 776160616.", "Collé : [TEL]."),
       ("Le Pro est à 49 000 F CFA, rendez-vous le 2026-11-03 pour 30 minutes", "Le Pro est à 49 000 F CFA, rendez-vous le 2026-11-03 pour 30 minutes")]
echecs = [i + 1 for i, (e, a) in enumerate(cas) if masquer_pii(e) != a]
tampon = io.StringIO(); journal = logging.getLogger("awa_test"); journal.handlers.clear(); journal.filters.clear()
journal.addHandler(logging.StreamHandler(tampon)); journal.addFilter(FiltrePII()); journal.warning("appel vers %s", "+221 77 616 06 16")
if "[TEL]" not in tampon.getvalue() or "616" in tampon.getvalue(): echecs.append("filtre de journalisation")
print("Tous les tests passent." if not echecs else f"À REVOIR : {echecs}")

**Limites** : une expression régulière **ne trouve pas les noms** (« Mamadou Ba ») ni les numéros **dits en lettres** (« soixante-dix-sept six cent seize »). Pour aller plus loin : **ne pas journaliser** les transcriptions complètes, ou utiliser un outil de détection d'entités. Et **masquer à l'écriture** ne remplace pas une **durée de conservation** limitée.

### 13.7 Atelier : déployer Awa sur un numéro de test et passer un appel sortant

**Avant de toucher au téléphone**, faites une **répétition générale** hors ligne : pour un appel sortant, on enchaîne les **contrôles** (`appel_autorise`) puis l'**appel**, dont l'**échec** est classé (`classer_echec_sip`).

In [ ]:
def repeter_appel_sortant(maintenant, numero, code_sip=None, liste_refus=frozenset()):
    """Répétition générale : ne compose aucun numéro. code_sip simule la réponse de l'opérateur (None = le correspondant décroche)."""
    ok, raison = appel_autorise(maintenant, numero, set(liste_refus), HORAIRES)
    if not ok:
        return f"appel NON lancé : {raison}"
    if code_sip is not None:
        return f"appel lancé, échec ({classer_echec_sip(code_sip)}) : libérer la tâche avec ctx.shutdown()"
    return "appel lancé, le correspondant décroche : attendre qu'il parle, puis Awa s'annonce comme IA"

for m, n, c in [(LUNDI_10H, "+221776160616", None), (LUNDI_10H, "+221776160616", 486), (LUNDI_10H, "+221776160616", 480),
                (LUNDI_10H, "+221776160616", 503), (DIMANCHE_10H, "+221776160616", None), (LUNDI_10H, "+33612345678", None)]:
    print(f"{n:15} code {str(c):5} -> {repeter_appel_sortant(m, n, c)}")

**Étapes de l'atelier** (sur **votre machine**, avec **votre numéro de test** : le vôtre ou celui d'un collègue **qui a accepté** d'être appelé) :

1. **Préparer** : un trunk SIP sortant chez un opérateur, son **identifiant** (`lk sip outbound list`), un **numéro de test**.
2. **Secrets** : créez `.env.production` (**hors dépôt**) avec `LIVEKIT_URL`, `LIVEKIT_API_KEY`, `LIVEKIT_API_SECRET`, `OPENAI_API_KEY` **si besoin**, `AWA_SIP_TRUNK_ID` et `AWA_AGENT_NAME`.
3. **Auditer** : lancez `rapport(...)` du 13.4 sur **votre vrai projet** (adaptez le dictionnaire) : **aucun échec** avant de continuer.
4. **Déployer en préproduction** : `lk agent create` puis `lk agent deploy --deployment staging` (13.1).
5. **Appeler** : `lk dispatch create --new-room --agent-name <AWA_AGENT_NAME> --metadata '{"phone_number": "+221..."}'`. Awa contrôle les horaires, compose, **attend** que vous parliez, puis **s'annonce comme IA**.
6. **Tester les échecs** : refusez l'appel, ne répondez pas, appelez hors horaires : relevez ce que `awa_agent.py` journalise.
7. **Vérifier les journaux** : **aucun numéro en clair**.

**Livrable** : la **sortie de la checklist** (tout en `ok`), un **extrait de journal masqué**, et **trois lignes** : ce qui a marché, ce qui a échoué, ce que vous changeriez avant la production. Si vous n'avez **pas d'accès téléphonique**, livrez la **répétition générale** et votre **plan** pas à pas.

`awa_agent.py` contient désormais l'**annonce de l'IA**, le **filtre de journaux**, le **contrôle des horaires** et l'**appel sortant** (variables `AWA_AGENT_NAME` et `AWA_SIP_TRUNK_ID`). Le transfert **réel** (module 10) et cet appel sortant **n'ont pas été testés avec un vrai opérateur** au moment de l'écriture : testez-les d'abord sur votre numéro.

### À retenir

- **Déployer** : LiveKit Cloud (`lk agent create`, `lk agent deploy`, préproduction puis `promote`), Pipecat Cloud (`pipecat cloud deploy`, réécriture nécessaire) ou **auto-hébergement** (contrôle total, exploitation à votre charge).
- **Appel sortant** : `create_sip_participant` avec `wait_until_answered`, `SipCallError` classé (**486 et 603** refus ; **408 et 480** pas de réponse ; **5xx** panne), **messagerie non détectée** par `wait_until_answered`.
- **Coût mensuel** = volume de minutes × (architecture + téléphonie) + fixe ; **calculez avec votre volume**.
- **Sécurité** : aucune clé dans le code, `.env` ignoré, une clé par environnement, **checklist exécutable** avant chaque déploiement.
- **Conformité** : **annoncer l'IA**, respecter le **refus**, **horaires** raisonnables, **conservation** limitée, **journaux masqués**. Tout cela est un **repère**, à **faire relire par le juridique**.

---
### Quiz du Module 13

**Q1.** Que se passe-t-il avec `wait_until_answered=True` quand le correspondant a une messagerie vocale ?
a) une erreur 486 est levée · b) l'appel est considéré comme décroché, car la messagerie répond, donc il faut une détection de répondeur à part · c) une erreur 408 est levée · d) l'appel est transféré à un humain

**Q2.** Où doit-on placer la clé de l'API d'un fournisseur ?
a) dans le code, si le dépôt est privé · b) dans le prompt d'Awa · c) dans le fichier README · d) dans une variable d'environnement ou un gestionnaire de secrets, jamais dans le dépôt

**Q3.** Un appel sortant échoue avec le code SIP 603. Que signifie-t-il ?
a) le correspondant a refusé l'appel · b) le correspondant n'a pas répondu · c) le trunk de l'opérateur est en panne · d) le numéro n'existe pas

**Q4.** Pourquoi masquer les numéros et les e-mails dans les journaux ?
a) pour que les journaux soient plus rapides · b) parce que les journaux sont illisibles sinon · c) parce que les journaux sont copiés et lus par plus de monde que le CRM : un numéro en clair y est une fuite possible · d) parce que LiveKit l'interdit

**Q5.** Comment traiter les repères du cours sur l'annonce de l'IA, le consentement, les horaires et la conservation ?
a) comme la loi sénégalaise, appliquée telle quelle · b) comme des repères de bonne pratique, à faire relire par le juridique avant la production · c) on peut les ignorer pour un test · d) ils ne concernent que les appels entrants

In [ ]:
mes_reponses_m13 = {1: "", 2: "", 3: "", 4: "", 5: ""}   # TODO : vos réponses

In [ ]:
def corriger_quiz_module13(reponses):
    attendu = {1: "b", 2: "d", 3: "a", 4: "c", 5: "b"}
    explications = {
        1: "Une messagerie répond comme un humain (200 OK) : wait_until_answered ne la détecte pas. Il faut une détection de répondeur.",
        2: "Les clés restent hors du code et du dépôt : variables d'environnement ou gestionnaire de secrets, une clé par environnement.",
        3: "603 (comme 486) : le correspondant a refusé. 408 et 480 : pas de réponse. 5xx : panne côté trunk ou opérateur.",
        4: "Les journaux circulent plus que le CRM. On masque à l'écriture, et on limite la durée de conservation.",
        5: "Ce sont des repères de bonne pratique, pas un avis juridique : le juridique relit avant la production (loi 2008-12 et CDP).",
    }
    score_quiz = 0
    for q in attendu:
        r = str(reponses.get(q, "")).strip().lower()
        ok = r == attendu[q]; score_quiz += ok
        print(f"Q{q} : {'correct' if ok else 'à revoir'} (votre réponse : {r or '-'}). {explications[q]}")
    print(f"\nScore : {score_quiz}/5")

corriger_quiz_module13(mes_reponses_m13)